# W04 - Salaries by Title & Country

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load the salary dataset with the correct date
salaries = pd.read_csv('Salaries_v4_202412161100.csv')
# salaries = pd.read_csv('Salaries_v3_202410141100.csv')
countries_coef = pd.read_csv('countries_salary_coef.csv')
salaries = salaries.drop_duplicates('id').reset_index()
salaries = salaries.drop('index',axis=1)

In [3]:
# Remove the specific rows that can cause inconsistency to the salary dataset
# Salaries_v3_202410141100.csv
salaries = salaries.drop(17343,axis=0)
## Salaries_v4_202412161100.csv
salaries = salaries.drop([14255,8691,8931],axis=0)

### Inıtial Info About Dataset

In [4]:
print("#### THIS SALARY DATASET CONTAINS {} ROWS & {} COLUMNS ####".format(salaries.shape[0], salaries.shape[1]))

#### THIS SALARY DATASET CONTAINS 19553 ROWS & 16 COLUMNS ####


In [5]:
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,NaN,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,NaN,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000


In [6]:
salaries.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19553 entries, 0 to 19556
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 19553 non-null  object 
 1   found_country      19553 non-null  object 
 2   title              19541 non-null  object 
 3   position           19553 non-null  object 
 4   employment_type    12396 non-null  object 
 5   company            16935 non-null  object 
 6   company_score      15738 non-null  float64
 7   edu_degrees        18999 non-null  object 
 8   edu_degrees_major  18622 non-null  object 
 9   working_year       19553 non-null  int64  
 10  education_score    14956 non-null  float64
 11  ms_counts          19553 non-null  int64  
 12  skill_counts       19553 non-null  int64  
 13  main_skills        19553 non-null  object 
 14  skills             19553 non-null  object 
 15  amount_usd         19553 non-null  int64  
dtypes: float64(2), int64(4

In [7]:
salaries.isnull().sum()

id                      0
found_country           0
title                  12
position                0
employment_type      7157
company              2618
company_score        3815
edu_degrees           554
edu_degrees_major     931
working_year            0
education_score      4597
ms_counts               0
skill_counts            0
main_skills             0
skills                  0
amount_usd              0
dtype: int64

In [8]:
round(salaries.describe(), 3)

,company_score,working_year,education_score,ms_counts,skill_counts,amount_usd
count,15738.000,19553.000,14956.000,19553.000,19553.000,19553.000
mean,8.249,11.450,8.401,15.738,18.778,163372.095
std,0.562,7.214,1.383,7.306,7.815,81116.532
min,3.800,0.000,1.600,1.000,1.000,21000.000
25%,8.100,6.000,7.500,11.000,15.000,102000.000
50%,8.400,10.000,8.600,16.000,20.000,150000.000
75%,8.600,15.000,9.700,20.000,20.000,210000.000
max,10.000,57.000,10.000,56.000,82.000,560000.000


### Info About Country Coefficients

In [9]:
# Show the countries' Turkish and English name with their salary coefficients
print(countries_coef[:20])

                        tr_name               en_name  salary_coefficient
0   Amerika Birleşik Devletleri         United States               1.000
1                        Kanada                Canada               0.769
2                        Monako                Monaco               0.766
3                    Lüksemburg            Luxembourg               0.766
4                       İrlanda               Ireland               0.765
5                        İsrail                Israel               0.763
6                       İzlanda               Iceland               0.750
7                       İsviçre           Switzerland               0.740
8                         Malta                 Malta               0.735
9                       Tayland              Thailand               0.729
10                        Katar                 Qatar               0.700
11    Birleşik Arap Emirlikleri  United Arab Emirates               0.700
12                       Norveç       

## SALARY INFO BY TITLES (HARDCODED)

In [10]:
### Count each identified title of all rows, and collect their corresponding salaries
### Titles indicating openness to work and opportunities
title_open_to_work = 0;  
### White collar titles
title_account_manager = 0;         salary_account_manager = [];         title_ai_eng = 0;               salary_ai_eng = [];  
title_app_dev_eng = 0;             salary_app_dev_eng = [];             title_applied_scientist = 0;    salary_applied_scientist = [];  
title_asic_design_eng = 0;         salary_asic_design_eng = [];         title_backend_dev = 0;          salary_backend_dev = [];  
title_bi_analyst = 0;              salary_bi_analyst = [];              title_bi_engineer = 0;          salary_bi_engineer = [];  
title_chemical_eng = 0;            salary_chemical_eng = [];            title_civil_eng = 0;            salary_civil_eng = [];  
title_cloud_eng = 0;               salary_cloud_eng = [];               title_computer_eng = 0;         salary_computer_eng = [];  
title_consultant = 0;              salary_consultant = [];              title_content_designer = 0;     salary_content_designer = [];  
title_content_marketer = 0;        salary_content_marketer = [];        title_content_strat = 0;        salary_content_strat = [];  
title_customer_eng = 0;            salary_customer_eng = [];            title_data_analyst = 0;         salary_data_analyst = [];  
title_data_architect = 0;          salary_data_architect = [];          title_data_eng = 0;             salary_data_eng = [];  
title_data_scientist = 0;          salary_data_scientist = [];          title_db_admin = 0;             salary_db_admin = [];  
title_db_eng = 0;                  salary_db_eng = [];                  title_design_eng = 0;           salary_design_eng = [];  
title_devops_eng = 0;              salary_devops_eng = [];              title_distinguished_eng = 0;    salary_distinguished_eng = [];  
title_dotnet_dev = 0;              salary_dotnet_dev = [];              title_ecommerce = 0;            salary_ecommerce = [];  
title_electric_electronic_eng = 0; salary_electric_electronic_eng = []; title_energy_analyst = 0;       salary_energy_analyst = [];  
title_environment_eng = 0;         salary_environment_eng = [];         title_financial_analyst = 0;    salary_financial_analyst = [];  
title_firmware_dev = 0;            salary_firmware_dev = [];            title_frontend_dev = 0;         salary_frontend_dev = [];  
title_fullstack_dev = 0;           salary_fullstack_dev = [];           title_graphic_designer = 0;     salary_graphic_designer = [];
title_hardware_eng = 0;            salary_hardware_eng = [];            title_inv_banking_analyst = 0;  salary_inv_banking_analyst = [];
title_it_architect_eng = 0;        salary_it_architect_eng = [];        title_java_dev = 0;             salary_java_dev = [];  
title_machine_learning_eng = 0;    salary_machine_learning_eng = [];    title_marketing_analyst = 0;    salary_marketing_analyst = [];  
title_mechanical_eng = 0;          salary_mechanical_eng = [];          title_mobile_dev = 0;           salary_mobile_dev = [];  
title_network_admin = 0;           salary_network_admin = [];           title_network_eng = 0;          salary_network_eng = [];  
title_net_sec_specialist = 0;      salary_net_sec_specialist = [];      title_operations_analyst = 0;   salary_operations_analyst = [];  
title_partner_spec = 0;            salary_partner_spec = [];            title_petroleum_eng = 0;        salary_petroleum_eng = [];  
title_platform_eng = 0;            salary_platform_eng = [];            title_process_eng = 0;          salary_process_eng = [];  
title_product_manager = 0;         salary_product_manager = [];         title_product_digital_mrkt = 0; salary_product_digital_mrkt = [];    
title_program_manager = 0;         salary_program_manager = [];         title_project_manager = 0;      salary_project_manager = [];  
title_python_dev = 0;              salary_python_dev = [];              title_qa_analyst = 0;           salary_qa_analyst = [];  
title_qa_engineer = 0;             salary_qa_engineer = [];             title_research_eng = 0;         salary_research_eng = [];  
title_sales_dev_repr = 0;          salary_sales_dev_repr = [];          title_sales_eng = 0;            salary_sales_eng = [];               
title_salesforce_dev = 0;          salary_salesforce_dev = [];          title_scrum_master = 0;         salary_scrum_master = [];  
title_security_eng = 0;            salary_security_eng = [];            title_seo_specialist = 0;       salary_seo_specialist = [];  
title_servicenow_dev = 0;          salary_servicenow_dev = [];          title_sharepoint_dev = 0;       salary_sharepoint_dev = [];  
title_software_eng = 0;            salary_software_eng = [];            title_solutions_architect = 0;  salary_solutions_architect = [];  
title_staff_eng = 0;               salary_staff_eng = [];               title_supply_chain_sol = 0;     salary_supply_chain_sol = [];  
title_support_eng = 0;             salary_support_eng = [];             title_system_admin = 0;         salary_system_admin = [];  
title_system_eng = 0;              salary_system_eng = [];              title_technical_analyst = 0;    salary_technical_analyst = [];  
title_test_eng = 0;                salary_test_eng = [];                title_ux_ui_designer = 0;       salary_ux_ui_designer = [];  
title_web_developer = 0;           salary_web_developer = [];  
### Blue collar titles
title_academic_advisor = 0;        salary_academic_advisor = [];        title_account_exec = 0;        salary_account_exec = [];  
title_account_strategist = 0;      salary_account_strategist = [];      title_accountant = 0;          salary_accountant = [];  
title_actuary = 0;                 salary_actuary = [];                 title_admission_counselor = 0; salary_admission_counselor = [];  
title_auditor = 0;                 salary_auditor = [];                 title_baker = 0;               salary_baker = [];  
title_barista = 0;                 salary_barista = [];                 title_bartender = 0;           salary_bartender = [];  
title_beauty_advisor = 0;          salary_beauty_advisor = [];          title_bookkeeper = 0;          salary_bookkeeper = [];  
title_claims_adjuster = 0;         salary_claims_adjuster = [];         title_compliance_officer = 0;  salary_compliance_officer = [];  
title_corporate_counsel = 0;       salary_corporate_counsel = [];       title_curriculum_dev = 0;      salary_curriculum_dev = [];  
title_customer_serv_man = 0;       salary_customer_serv_man = [];       title_delivery_driver = 0;     salary_delivery_driver = [];  
title_dental_asst = 0;             salary_dental_asst = [];             title_electrician = 0;         salary_electrician = [];  
title_executive_asst = 0;          salary_executive_asst = [];          title_food_scientist = 0;      salary_food_scientist = [];  
title_help_desk_support_anly = 0;  salary_help_desk_support_anly = [];  title_inside_sales = 0;        salary_inside_sales = [];  
title_insurance_agent = 0;         salary_insurance_agent = [];         title_interaction_design = 0;  salary_interaction_design = [];  
title_librarian = 0;               salary_librarian = [];               title_medical_assistant = 0;   salary_medical_assistant = [];  
title_merchandiser = 0;            salary_merchandiser = [];            title_nurse = 0;               salary_nurse = [];  
title_office_admin = 0;            salary_office_admin = [];            title_outside_sales = 0;       salary_outside_sales = [];  
title_paralegal = 0;               salary_paralegal = [];               title_paramedic = 0;           salary_paramedic = [];  
title_police_officer = 0;          salary_police_officer = [];          title_policy_analyst = 0;      salary_policy_analyst = [];  
title_principal = 0;               salary_principal = [];               title_program_coord = 0;       salary_program_coord = [];  
title_psychologist = 0;            salary_psychologist = [];            title_real_estate_agent = 0;   salary_real_estate_agent = [];  
title_recruit = 0;                 salary_recruit = [];                 title_sales_assoc = 0;         salary_sales_assoc = [];  
title_school_counselor = 0;        salary_school_counselor = [];        title_secretary = 0;           salary_secretary = [];  
title_security_officer = 0;        salary_security_officer = [];        title_social_worker = 0;       salary_social_worker = [];  
title_supervisor = 0;              salary_supervisor = [];              title_talent_asst = 0;         salary_talent_asst = [];  
title_teacher = 0;                 salary_teacher = [];                 title_technician = 0;          salary_technician = [];  
title_therapist = 0;               salary_therapist = [];               title_touring_asst = 0;        salary_touring_asst = [];  
title_transport_planner = 0;       salary_transport_planner = [];       title_underwriter = 0;         salary_underwriter = [];  
title_urban_planner = 0;           salary_urban_planner = [];           title_vice_president = 0;      salary_vice_president = [];  
title_welder = 0;                  salary_welder = [];  
# Other titles that do not fit well in appropriate titles above
title_student = 0;     salary_student = [];     title_developer = 0;     salary_developer = [];  
title_engineer = 0;    salary_engineer = [];    title_server = 0;        salary_server = [];  
title_assistant = 0;   salary_assistant = [];   title_market = 0;        salary_market = [];  
title_analytics = 0;   salary_analytics = [];   title_analyst = 0;       salary_analyst = [];  
title_designer = 0;    salary_designer = [];  
# Uncategorized and null number of titles
title_null = 0;  title_uncategorized = 0;  title_uncategorized_examples = [];

In [11]:
all_titles = salaries['title'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_titles)):
    title = all_titles[i];   salary = all_salaries[i]
    if type(title) == float:   title_null += 1;   continue   ### NULL TITLES ###
    title = title.lower()
    ### OPEN TO WORK ###
    if 'open to work' in title or 'open to opportunities' in title or 'open to new opportunities' in title \
      or 'open to entry level' in title or 'looking for a new opportunity' in title or 'employment opportunity' in title \
      or 'experience with every given opportunity' in title or 'looking for new opportunities' in title \
      or 'looking for new carrier' in title or 'looking for opportunities' in title or 'looking for a new path' in title \
      or 'looking for a long term career' in title or 'looking for new recruiting opportunities' in title \
      or 'looking for professional opportunities' in title or 'looking for a full time position' in title \
      or 'looking for writing opportunities' in title or 'job searching' in title or 'seeking new opportunities' in title \
      or 'seeking opportunities' in title:
        title_open_to_work += 1
    ### WHITE COLLAR ###
    elif 'account manager' in title:   
        title_account_manager += 1;   salary_account_manager.append(salary);
    elif 'ai engineering' in title or 'ai engineer' in title or 'ai frameworks engineer' in title \
      or 'ai research engineer' in title or 'ai model designer' in title or 'ai product designer' in title or 'ai designer' in title:
        title_ai_eng += 1;   salary_ai_eng.append(salary);
    elif 'application developer' in title or 'app developer' in title or 'development engineer' in title \
      or 'application engineer' in title or 'applications engineer' in title:
        title_app_dev_eng += 1;   salary_app_dev_eng.append(salary);
    elif 'applied scientist' in title:   
        title_applied_scientist += 1;   salary_applied_scientist.append(salary);
    elif 'asic design engineer' in title:   
        title_asic_design_eng += 1;   salary_asic_design_eng.append(salary);
    elif 'backend developer' in title or 'back end developer' in title or 'back-end developer' in title \
      or 'backend engineer' in title or 'back end engineer' in title or 'back-end engineer' in title:   
        title_backend_dev += 1;   salary_backend_dev.append(salary);
    elif 'business analyst' in title or 'business intelligence analyst' in title or 'business technology analyst' in title \
      or 'business systems analyst' in title or 'business strategy analyst' in title or 'bi analyst' in title \
      or 'business analytics' in title or ('business' in title and 'analyst' in title):
        title_bi_analyst += 1;   salary_bi_analyst.append(salary);
    elif 'business intelligence engineer' in title or 'bi engineer' in title or 'bi developer' in title \
      or 'business intelligence developer' in title or 'bi and analytics engineer' in title or 'business intelligence' in title:
        title_bi_engineer += 1;   salary_bi_engineer.append(salary);
    elif 'chemical engineer' in title:   
        title_chemical_eng += 1;   salary_chemical_eng.append(salary);
    elif 'civil engineer' in title or 'civil design engineer' in title or 'cilvil engineer' in title \
      or 'civil/structural/transportation engineer' in title:
        title_civil_eng += 1;   salary_civil_eng.append(salary);
    elif 'cloud engineer' in title or 'cloud infrastructure engineer' in title or 'cloud and innovation engineer' in title \
      or 'cloud architect' in title:
        title_cloud_eng += 1;   salary_cloud_eng.append(salary);
    elif 'computer engineer' in title:   
        title_computer_eng += 1;   salary_computer_eng.append(salary);
    elif 'consultant' in title:   
        title_consultant += 1;   salary_consultant.append(salary);
    elif 'content designer' in title or 'content developer' in title or 'content producer' in title:   
        title_content_designer += 1;   salary_content_designer.append(salary);
    elif 'content marketing' in title or 'content marketer' in title:   
        title_content_marketer += 1;   salary_content_marketer.append(salary);
    elif 'content strategist' in title:   
        title_content_strat += 1;   salary_content_strat.append(salary);
    elif 'customer engineer' in title:   
        title_customer_eng += 1;   salary_customer_eng.append(salary);
    elif 'data analyst' in title or 'data insights analyst' in title or 'data and systems analyst' in title \
      or 'data intelligence analyst' in title or 'data quality analyst' in title or 'data solutions analyst' in title \
      or 'data analytics' in title or ('data' in title and 'analyst' in title) or ('data' in title and 'analytics' in title):   
        title_data_analyst += 1;   salary_data_analyst.append(salary);
    elif 'data architect' in title:   
        title_data_architect += 1;   salary_data_architect.append(salary);
    elif 'data engineer' in title:   
        title_data_eng += 1;   salary_data_eng.append(salary);
    elif 'data scientist' in title or 'data science' in title:   
        title_data_scientist += 1;   salary_data_scientist.append(salary);
    elif 'database administrator' in title or 'database administration' in title or 'dba' in title:   
        title_db_admin += 1;   salary_db_admin.append(salary);
    elif 'database engineer' in title:  
        title_db_eng += 1;   salary_db_eng.append(salary);
    elif 'design engineer' in title or 'design verification engineer' in title or 'design/debug engineer' in title:   
        title_design_eng += 1;   salary_design_eng.append(salary);
    elif 'devops engineer' in title or 'devsecops engineer' in title or 'devops' in title:   
        title_devops_eng += 1;   salary_devops_eng.append(salary);
    elif 'distinguished engineer' in title:   
        title_distinguished_eng += 1;   salary_distinguished_eng.append(salary);
    elif 'dotnet developer' in title or '.net developer' in title:   
        title_dotnet_dev += 1;   salary_dotnet_dev.append(salary);
    elif 'ecommerce' in title or 'e-commerce' in title:   
        title_ecommerce += 1;   salary_ecommerce.append(salary);
    elif 'electrical engineer' in title or 'electronic engineer' in title or 'electronics engineer' in title:   
        title_electric_electronic_eng += 1;   salary_electric_electronic_eng.append(salary);
    elif 'energy analyst' in title or 'energy & chemicals' in title:   
        title_energy_analyst += 1;   salary_energy_analyst.append(salary);
    elif 'environmental engineer' in title or 'environmental scientist' in title or 'environmental compliance engineer' in title:
        title_environment_eng += 1;   salary_environment_eng.append(salary);
    elif 'financial analyst' in title or 'finance project analyst' in title or 'financial analytics' in title \
      or 'finance analyst' in title or 'finance professional' in title:
        title_financial_analyst += 1;   salary_financial_analyst.append(salary);
    elif 'firmware developer' in title or 'firmware engineer' in title:   
        title_firmware_dev += 1;   salary_firmware_dev.append(salary);
    elif 'frontend developer' in title or 'front-end developer' in title or 'front end developer' in title \
      or 'frontend engineer' in title or 'front-end engineer' in title or 'front end engineer' in title \
      or 'javascript developer' in title or 'react developer' in title:   
        title_frontend_dev += 1;   salary_frontend_dev.append(salary);
    elif 'fullstack developer' in title or 'full stack developer' in title or 'full-stack developer' in title \
      or 'fullstack engineer' in title or 'full stack engineer' in title or 'full-stack engineer' in title:   
        title_fullstack_dev += 1;   salary_fullstack_dev.append(salary);
    elif 'graphic designer' in title:   
        title_graphic_designer += 1;   salary_graphic_designer.append(salary);
    elif 'hardware engineer' in title or 'hardware design engineer' in title:   
        title_hardware_eng += 1;   salary_hardware_eng.append(salary);
    elif 'investment banking analyst' in title:   
        title_inv_banking_analyst += 1;   salary_inv_banking_analyst.append(salary);
    elif 'it engineer' in title or 'it support engineer' in title or 'information technology engineer' in title \
      or 'it architect' in title:
        title_it_architect_eng += 1;   salary_it_architect_eng.append(salary);
    elif 'java developer' in title or 'java/guidewire developer' in title \
      or 'java full stack aws developer' in title or 'java microservices developer' in title or 'java/j2ee developer' in title:   
        title_java_dev += 1;   salary_java_dev.append(salary);
    elif 'machine learning engineer' in title or 'machine learning scientist' in title or 'ml engineer' in title \
      or 'machine learning' in title or 'deep learning engineer' in title or title.startswith('ml'):
        title_machine_learning_eng += 1;   salary_machine_learning_eng.append(salary);
    elif 'marketing analyst' in title or 'market research analyst' in title or 'market analyst' in title \
      or 'marketing specialist' in title or 'marketing strategist' in title or 'marketing analytics' in title \
      or 'marketing execution analyst' in title or 'marketing executive' in title or 'market expert' in title \
      or 'marketing professional' in title or 'growth marketing' in title or 'brand marketing' in title \
      or 'global marketing' in title or 'salesforce marketing' in title or 'lifecycle marketing' in title:
        title_marketing_analyst += 1;   salary_marketing_analyst.append(salary);
    elif 'mechanical engineer' in title or 'mechanical display engineer' in title or 'mechanical metrology engineer' in title:
        title_mechanical_eng += 1;   salary_mechanical_eng.append(salary);
    elif 'mobile developer' in title or 'ios developer' in title or 'android developer' in title:   
        title_mobile_dev += 1;   salary_mobile_dev.append(salary);
    elif 'network administrator' in title or 'network administration' in title:   
        title_network_admin += 1;   salary_network_admin.append(salary);
    elif 'network engineer' in title or 'networks engineer' in title or 'network development engineer' in title \
      or 'network.engineer' in title or 'infrastructure engineer' in title or 'infrastructure services engineer' in title \
      or 'network consulting engineer' in title:
        title_network_eng += 1;   salary_network_eng.append(salary);
    elif 'network specialist' in title or 'security specialist' in title:   
        title_net_sec_specialist += 1;   salary_net_sec_specialist.append(salary);
    elif 'operations analyst' in title or 'operations engineer' in title:  
        title_operations_analyst += 1;   salary_operations_analyst.append(salary);
    elif 'partner specialist' in title:   
        title_partner_spec += 1;   salary_partner_spec.append(salary);
    elif 'petroleum engineer' in title:   
        title_petroleum_eng += 1;   salary_petroleum_eng.append(salary);
    elif 'platform engineer' in title or 'platform developer' in title or 'plaform engineer' in title:   
        title_platform_eng += 1;   salary_platform_eng.append(salary);
    elif 'process engineer' in title:   
        title_process_eng += 1;   salary_process_eng.append(salary);
    elif 'product manager' in title or 'product development engineer' in title or 'product marketing manager' in title \
      or 'product analytics' in title:
        title_product_manager += 1;   salary_product_manager.append(salary);
    elif 'product marketing' in title or 'product marketer' in title or 'digital marketing' in title \
      or 'digital marketer' in title or 'digital advertising' in title:   
        title_product_digital_mrkt += 1;   salary_product_digital_mrkt.append(salary);
    elif 'program manager' in title:   
        title_program_manager += 1;   salary_program_manager.append(salary);
    elif 'project manager' in title or 'project mananger' in title:   
        title_project_manager += 1;   salary_project_manager.append(salary);
    elif 'python developer' in title:   
        title_python_dev += 1;   salary_python_dev.append(salary);
    elif 'qa analyst' in title:   
        title_qa_analyst += 1;   salary_qa_analyst.append(salary);
    elif 'qa engineer' in title or 'quality assurance' in title or 'quality engineer' in title \
      or 'qa automation engineer' in title or 'qa lead automation engineer' in title or 'qa mobile tester engineer' in title \
      or 'qa lead engineer' in title or 'qe engineering' in title or 'automation verification engineer' in title:
        title_qa_engineer += 1;   salary_qa_engineer.append(salary);
    elif 'research engineer' in title:   
        title_research_eng += 1;   salary_research_eng.append(salary);
    elif 'sales development representative' in title:   
        title_sales_dev_repr += 1;   salary_sales_dev_repr.append(salary);
    elif 'sales engineer' in title:   
        title_sales_eng += 1;   salary_sales_eng.append(salary);
    elif 'salesforce developer' in title or 'salesforce admin and developer' in title:   
        title_salesforce_dev += 1;   salary_salesforce_dev.append(salary);
    elif 'scrum master' in title:   
        title_scrum_master += 1;   salary_scrum_master.append(salary);
    elif 'security engineer' in title:   
        title_security_eng += 1;   salary_security_eng.append(salary);
    elif 'seo specialist' in title or 'seo analyst' in title:   
        title_seo_specialist += 1;   salary_seo_specialist.append(salary);
    elif 'servicenow developer' in title or 'servicenow grc/irm-secops architect' in title:   
        title_servicenow_dev += 1;   salary_servicenow_dev.append(salary);
    elif 'sharepoint developer' in title:   
        title_sharepoint_dev += 1;   salary_sharepoint_dev.append(salary);
    elif 'software engineer' in title or 'software development engineer' in title or 'software developer' in title \
      or 'software / firmware engineer' in title or 'softwear developer' in title or 'sofware developer' in title \
      or 'software dev engineer' in title or 'software devlopment engineer' in title \
      or 'software application development engineer' in title or 'software applications development engineer' in title \
      or 'software development emulator' in title or title.startswith('sde'):   
        title_software_eng += 1;   salary_software_eng.append(salary);
    elif 'solution architect' in title or 'solutions architect' in title or 'solution developer' in title \
      or 'solution engineer' in title or 'solutions engineer' in title:
        title_solutions_architect += 1;   salary_solutions_architect.append(salary);
    elif 'staff engineer' in title:   
        title_staff_eng += 1;   salary_staff_eng.append(salary);
    elif 'supply chain solutions' in title or 'supply chain planning' in title or 'supply chain analyst' in title \
      or 'supply chain program analyst' in title:
        title_supply_chain_sol += 1;   salary_supply_chain_sol.append(salary);
    elif 'support engineer' in title:   
        title_support_eng += 1;   salary_support_eng.append(salary);
    elif 'system administrator' in title or 'systems administrator' in title:   
        title_system_admin += 1;   salary_system_admin.append(salary);
    elif 'system engineer' in title or 'systems engineer' in title or 'system admin/engineer' in title \
      or 'system development engineer' in title or 'systems development engineer' in title:   
        title_system_eng += 1;   salary_system_eng.append(salary);
    elif 'technical analyst' in title or 'technical solution analyst' in title or 'technical solutions analyst' in title:
        title_technical_analyst += 1;   salary_technical_analyst.append(salary);
    elif 'test engineer' in title or 'test automation engineer' in title or 'test automation developer' in title \
      or 'automation engineer' in title or 'test development engineer' in title or 'test r&d engineer' in title \
      or 'test r & d engineer' in title or 'test product development engineer' in title or 'software test' in title:
        title_test_eng += 1;   salary_test_eng.append(salary);
    elif 'user experience designer' in title or 'user experience, visual designer' in title \
      or 'user experience researcher & designer' in title or 'user experience engineer' in title \
      or 'user experience (ux) designer' in title or 'ux/ui designer' in title or 'ux ui designer' in title \
      or 'user experience researcher' in title or 'user experience & brand designer' in title \
      or 'ux visual & motion designer' in title or 'ui/ux visual designer' in title or 'ux/visual designer' in title \
      or 'ux designer' in title or 'ux engineer' in title or 'ux/ ui designer' in title or 'product designer' in title \
      or 'user interface developer' in title or 'ui developer' in title:   
        title_ux_ui_designer += 1;   salary_ux_ui_designer.append(salary);
    elif 'web developer' in title or 'web production developer' in title:   
        title_web_developer += 1;   salary_web_developer.append(salary);
    ### BLUE COLLAR ###
    elif 'academic advisor' in title or 'academic advising' in title:   
        title_academic_advisor += 1;   salary_academic_advisor.append(salary);
    elif 'account executive' in title:   
        title_account_exec += 1;   salary_account_exec.append(salary);
    elif 'account strategist' in title:   
        title_account_strategist += 1;   salary_account_strategist.append(salary);
    elif 'accountant' in title:   
        title_accountant += 1;   salary_accountant.append(salary);
    elif 'actuary' in title:   
        title_actuary += 1;   salary_actuary.append(salary);
    elif 'admission counselor' in title or 'admissions counselor' in title:   
        title_admission_counselor += 1;   salary_admission_counselor.append(salary);
    elif 'auditor' in title or 'audit associate' in title:   
        title_auditor += 1;   salary_auditor.append(salary);
    elif 'baker' in title:   
        title_baker += 1;   salary_baker.append(salary);
    elif 'barista' in title:   
        title_barista += 1;   salary_barista.append(salary);
    elif 'bartender' in title:   
        title_bartender += 1;   salary_bartender.append(salary);
    elif 'bookkeeper' in title:   
        title_bookkeeper += 1;   salary_bookkeeper.append(salary);
    elif 'beauty advisor' in title:   
        title_beauty_advisor += 1;   salary_beauty_advisor.append(salary);
    elif 'claims adjuster' in title:   
        title_claims_adjuster += 1;   salary_claims_adjuster.append(salary);
    elif 'compliance officer' in title or 'compliance risk officer' in title or 'compliance/operations officer' in title \
      or 'compliance risk management officer' in title:
        title_compliance_officer += 1;   salary_compliance_officer.append(salary);
    elif 'corporate counsel' in title:   
        title_corporate_counsel += 1;   salary_corporate_counsel.append(salary);
    elif 'curriculum developer' in title:   
        title_curriculum_dev += 1;   salary_curriculum_dev.append(salary);
    elif 'customer service manager' in title:   
        title_customer_serv_man += 1;   salary_customer_serv_man.append(salary);
    elif 'delivery driver' in title:   
        title_delivery_driver += 1;   salary_delivery_driver.append(salary);
    elif 'dental assistant' in title or 'dental assisting' in title:   
        title_dental_asst += 1;    salary_dental_asst.append(salary);
    elif 'electrician' in title:   
        title_electrician += 1;   salary_electrician.append(salary);
    elif 'executive assistant' in title or 'executive & administrative assistant' in title:   
        title_executive_asst += 1;   salary_executive_asst.append(salary);
    elif 'food scientist' in title:   
        title_food_scientist += 1;   salary_food_scientist.append(salary);
    elif 'help desk support' in title or 'help desk analyst' in title:   
        title_help_desk_support_anly += 1;   salary_help_desk_support_anly.append(salary);
    elif 'inside sales' in title:   
        title_inside_sales += 1;   salary_inside_sales.append(salary);
    elif 'insurance agent' in title:   
        title_insurance_agent += 1;   salary_insurance_agent.append(salary);
    elif 'interaction designer' in title:   
        title_interaction_design += 1;   salary_interaction_design.append(salary);
    elif 'librarian' in title:   
        title_librarian += 1;   salary_librarian.append(salary);
    elif 'medical assistant' in title:   
        title_medical_assistant += 1;   salary_medical_assistant.append(salary);
    elif 'merchandiser' in title:   
        title_merchandiser += 1;   salary_merchandiser.append(salary);
    elif 'nurse' in title:   
        title_nurse += 1;   salary_nurse.append(salary);
    elif 'office administrator' in title:   
        title_office_admin += 1;   salary_office_admin.append(salary);
    elif 'outside sales' in title:   
        title_outside_sales += 1;   salary_outside_sales.append(salary);
    elif 'paralegal' in title:   
        title_paralegal += 1;   salary_paralegal.append(salary);
    elif 'paramedic' in title:   
        title_paramedic += 1;   salary_paramedic.append(salary);
    elif 'police officer' in title or 'police official' in title:   
        title_police_officer += 1;   salary_police_officer.append(salary);
    elif 'policy analyst' in title:   
        title_policy_analyst += 1;   salary_policy_analyst.append(salary);
    elif 'principal' in title:   
        title_principal += 1;   salary_principal.append(salary);
    elif 'program coordinator' in title:   
        title_program_coord += 1;   salary_program_coord.append(salary);
    elif 'psychologist' in title:   
        title_psychologist += 1;   salary_psychologist.append(salary);
    elif 'real estate agent' in title:   
        title_real_estate_agent += 1;   salary_real_estate_agent.append(salary);
    elif 'recruit' in title:   
        title_recruit += 1;   salary_recruit.append(salary);
    elif 'sales associate' in title or 'sales representative' in title:   
        title_sales_assoc += 1;   salary_sales_assoc.append(salary);
    elif 'school counselor' in title:  
        title_school_counselor += 1;   salary_school_counselor.append(salary);
    elif 'secretary' in title:   
        title_secretary += 1;   salary_secretary.append(salary);
    elif 'security officer' in title or 'security guard' in title:   
        title_security_officer += 1;   salary_security_officer.append(salary);
    elif 'social worker' in title or 'lmsw' in title:   
        title_social_worker += 1;   salary_social_worker.append(salary);
    elif 'supervisor' in title:   
        title_supervisor += 1;   salary_supervisor.append(salary);
    elif 'talent assistant' in title:   
        title_talent_asst += 1;   salary_talent_asst.append(salary);
    elif 'teacher' in title:   
        title_teacher += 1;   salary_teacher.append(salary);
    elif 'technician' in title:   
        title_technician += 1;   salary_technician.append(salary);
    elif 'therapist' in title:   
        title_therapist += 1;   salary_therapist.append(salary);
    elif 'touring assistant' in title:   
        title_touring_asst += 1;   salary_touring_asst.append(salary);
    elif 'transportation planner' in title:   
        title_transport_planner += 1;   salary_transport_planner.append(salary);
    elif 'underwriter' in title:   
        title_underwriter += 1;   salary_underwriter.append(salary);
    elif 'urban planner' in title:   
        title_urban_planner += 1;   salary_urban_planner.append(salary);
    elif 'vice president' in title:   
        title_vice_president += 1;   salary_vice_president.append(salary);
    elif 'welder' in title:   
        title_welder += 1;   salary_welder.append(salary);
    ### OTHER NON-FITTING TITLES ###
    elif 'student' in title:     title_student += 1;      salary_student.append(salary);
    elif 'developer' in title:   title_developer += 1;    salary_developer.append(salary);
    elif 'engineer' in title:    title_engineer += 1;     salary_engineer.append(salary);
    elif 'server' in title:      title_server += 1;       salary_server.append(salary);
    elif 'assistant' in title:   title_assistant += 1;    salary_assistant.append(salary);
    elif 'market' in title:      title_market += 1;       salary_market.append(salary);
    elif 'analytics' in title:   title_analytics += 1;    salary_analytics.append(salary);
    elif 'analyst' in title:     title_analyst += 1;      salary_analyst.append(salary);
    elif 'designer' in title:    title_designer += 1;     salary_designer.append(salary);
    ### UNCATEGORIZED TITLES ###
    else:   title_uncategorized += 1;   title_uncategorized_examples.append(title)

In [12]:
print("###### {:4} NULL TITLES ######".format(title_null))
print("###### {:4} TITLES OPEN TO WORK / OPPORTUNITIES ######".format(title_open_to_work))
print("###### WHITE COLLAR TITLES ############ AVG ######## MIN ######## MAX ######")
print("{:4} ACCOUNT MANAGER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_account_manager, 
    np.array(salary_account_manager).mean(), np.array(salary_account_manager).min(), np.array(salary_account_manager).max()))
print("{:4} AI ENGINEER                  | {:10.2f} | {:10.2f} | {:10.2f}".format(title_ai_eng, 
    np.array(salary_ai_eng).mean(), np.array(salary_ai_eng).min(), np.array(salary_ai_eng).max()))
print("{:4} APPLICATION DEV. / ENGINEER  | {:10.2f} | {:10.2f} | {:10.2f}".format(title_app_dev_eng, 
    np.array(salary_app_dev_eng).mean(), np.array(salary_app_dev_eng).min(), np.array(salary_app_dev_eng).max()))
print("{:4} APPLIED SCIENTIST            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_applied_scientist, 
    np.array(salary_applied_scientist).mean(), np.array(salary_applied_scientist).min(), np.array(salary_applied_scientist).max()))
print("{:4} ASIC DESIGN ENGINEER         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_asic_design_eng, 
    np.array(salary_asic_design_eng).mean(), np.array(salary_asic_design_eng).min(), np.array(salary_asic_design_eng).max()))
print("{:4} BACKEND DEVELOPER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_backend_dev, 
    np.array(salary_backend_dev).mean(), np.array(salary_backend_dev).min(), np.array(salary_backend_dev).max()))
print("{:4} BUSINESS ANALYST             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_bi_analyst, 
    np.array(salary_bi_analyst).mean(), np.array(salary_bi_analyst).min(), np.array(salary_bi_analyst).max()))
print("{:4} BUSINESS INTELLIGENCE ENG.   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_bi_engineer, 
    np.array(salary_bi_engineer).mean(), np.array(salary_bi_engineer).min(), np.array(salary_bi_engineer).max()))
print("{:4} CHEMICAL ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_chemical_eng, 
    np.array(salary_chemical_eng).mean(), np.array(salary_chemical_eng).min(), np.array(salary_chemical_eng).max()))
print("{:4} CIVIL ENGINEER               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_civil_eng, 
    np.array(salary_civil_eng).mean(), np.array(salary_civil_eng).min(), np.array(salary_civil_eng).max()))
print("{:4} CLOUD ENGINEER               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_cloud_eng, 
    np.array(salary_cloud_eng).mean(), np.array(salary_cloud_eng).min(), np.array(salary_cloud_eng).max()))
print("{:4} COMPUTER ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_computer_eng, 
    np.array(salary_computer_eng).mean(), np.array(salary_computer_eng).min(), np.array(salary_computer_eng).max()))
print("{:4} CONSULTANT                   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_consultant, 
    np.array(salary_consultant).mean(), np.array(salary_consultant).min(), np.array(salary_consultant).max()))
print("{:4} CONTENT DESIGNER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_content_designer, 
    np.array(salary_content_designer).mean(), np.array(salary_content_designer).min(), np.array(salary_content_designer).max()))
print("{:4} CONTENT MARKETER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_content_marketer, 
    np.array(salary_content_marketer).mean(), np.array(salary_content_marketer).min(), np.array(salary_content_marketer).max()))
print("{:4} CONTENT STRATEGIST           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_content_strat, 
    np.array(salary_content_strat).mean(), np.array(salary_content_strat).min(), np.array(salary_content_strat).max()))
print("{:4} CUSTOMER ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_customer_eng, 
    np.array(salary_customer_eng).mean(), np.array(salary_customer_eng).min(), np.array(salary_customer_eng).max()))
print("{:4} DATA ANALYST                 | {:10.2f} | {:10.2f} | {:10.2f}".format(title_data_analyst, 
    np.array(salary_data_analyst).mean(), np.array(salary_data_analyst).min(), np.array(salary_data_analyst).max()))
print("{:4} DATA ARCHITECT               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_data_architect, 
    np.array(salary_data_architect).mean(), np.array(salary_data_architect).min(), np.array(salary_data_architect).max()))
print("{:4} DATA ENGINEER                | {:10.2f} | {:10.2f} | {:10.2f}".format(title_data_eng, 
    np.array(salary_data_eng).mean(), np.array(salary_data_eng).min(), np.array(salary_data_eng).max()))
print("{:4} DATA SCIENTIST               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_data_scientist, 
    np.array(salary_data_scientist).mean(), np.array(salary_data_scientist).min(), np.array(salary_data_scientist).max()))
print("{:4} DATABASE ADMINISTRATOR       | {:10.2f} | {:10.2f} | {:10.2f}".format(title_db_admin, 
    np.array(salary_db_admin).mean(), np.array(salary_db_admin).min(), np.array(salary_db_admin).max()))
print("{:4} DATABASE ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_db_eng, 
    np.array(salary_db_eng).mean(), np.array(salary_db_eng).min(), np.array(salary_db_eng).max()))
print("{:4} DESIGN ENGINEER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_design_eng, 
    np.array(salary_design_eng).mean(), np.array(salary_design_eng).min(), np.array(salary_design_eng).max()))
print("{:4} DEVOPS ENGINEER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_devops_eng, 
    np.array(salary_devops_eng).mean(), np.array(salary_devops_eng).min(), np.array(salary_devops_eng).max()))
print("{:4} DISTINGUISHED ENGINEER       | {:10.2f} | {:10.2f} | {:10.2f}".format(title_distinguished_eng, 
    np.array(salary_distinguished_eng).mean(), np.array(salary_distinguished_eng).min(), np.array(salary_distinguished_eng).max()))
print("{:4} DOTNET (.NET) DEVELOPER      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_dotnet_dev, 
    np.array(salary_dotnet_dev).mean(), np.array(salary_dotnet_dev).min(), np.array(salary_dotnet_dev).max()))
print("{:4} E-COMMERCE                   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_ecommerce, 
    np.array(salary_ecommerce).mean(), np.array(salary_ecommerce).min(), np.array(salary_ecommerce).max()))
print("{:4} ELECTRICAL / ELECTRONIC ENG. | {:10.2f} | {:10.2f} | {:10.2f}".format(title_electric_electronic_eng, 
    np.array(salary_electric_electronic_eng).mean(), np.array(salary_electric_electronic_eng).min(), np.array(salary_electric_electronic_eng).max()))
print("{:4} ENERGY ANALYST               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_energy_analyst, 
    np.array(salary_energy_analyst).mean(), np.array(salary_energy_analyst).min(), np.array(salary_energy_analyst).max()))
print("{:4} ENVIRONMENTAL ENGINEER       | {:10.2f} | {:10.2f} | {:10.2f}".format(title_environment_eng, 
    np.array(salary_environment_eng).mean(), np.array(salary_environment_eng).min(), np.array(salary_environment_eng).max()))
print("{:4} FINANCIAL ANALYST            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_financial_analyst, 
    np.array(salary_financial_analyst).mean(), np.array(salary_financial_analyst).min(), np.array(salary_financial_analyst).max()))
print("{:4} FIRMWARE DEVELOPER           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_firmware_dev, 
    np.array(salary_firmware_dev).mean(), np.array(salary_firmware_dev).min(), np.array(salary_firmware_dev).max()))
print("{:4} FRONTEND DEVELOPER           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_frontend_dev, 
    np.array(salary_frontend_dev).mean(), np.array(salary_frontend_dev).min(), np.array(salary_frontend_dev).max()))
print("{:4} FULLSTACK DEVELOPER          | {:10.2f} | {:10.2f} | {:10.2f}".format(title_fullstack_dev, 
    np.array(salary_fullstack_dev).mean(), np.array(salary_fullstack_dev).min(), np.array(salary_fullstack_dev).max()))
print("{:4} GRAPHIC DESIGNER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_graphic_designer, 
    np.array(salary_graphic_designer).mean(), np.array(salary_graphic_designer).min(), np.array(salary_graphic_designer).max()))
print("{:4} HARDWARE ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_hardware_eng, 
    np.array(salary_hardware_eng).mean(), np.array(salary_hardware_eng).min(), np.array(salary_hardware_eng).max()))
print("{:4} INVESTMENT BANKING ANALYST   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_inv_banking_analyst, 
    np.array(salary_inv_banking_analyst).mean(), np.array(salary_inv_banking_analyst).min(), np.array(salary_inv_banking_analyst).max()))
print("{:4} IT ARCHITECT / ENGINEER      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_it_architect_eng, 
    np.array(salary_it_architect_eng).mean(), np.array(salary_it_architect_eng).min(), np.array(salary_it_architect_eng).max()))
print("{:4} JAVA DEVELOPER               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_java_dev, 
    np.array(salary_java_dev).mean(), np.array(salary_java_dev).min(), np.array(salary_java_dev).max()))
print("{:4} MACHINE LEARNING ENGINEER    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_machine_learning_eng, 
    np.array(salary_machine_learning_eng).mean(), np.array(salary_machine_learning_eng).min(), np.array(salary_machine_learning_eng).max()))
print("{:4} MARKETING ANALYST            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_marketing_analyst, 
    np.array(salary_marketing_analyst).mean(), np.array(salary_marketing_analyst).min(), np.array(salary_marketing_analyst).max()))
print("{:4} MECHANICAL ENGINEER          | {:10.2f} | {:10.2f} | {:10.2f}".format(title_mechanical_eng, 
    np.array(salary_mechanical_eng).mean(), np.array(salary_mechanical_eng).min(), np.array(salary_mechanical_eng).max()))
print("{:4} MOBILE DEVELOPER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_mobile_dev, 
    np.array(salary_mobile_dev).mean(), np.array(salary_mobile_dev).min(), np.array(salary_mobile_dev).max()))
print("{:4} NETWORK ADMINISTRATOR        | {:10.2f} | {:10.2f} | {:10.2f}".format(title_network_admin, 
    np.array(salary_network_admin).mean(), np.array(salary_network_admin).min(), np.array(salary_network_admin).max()))
print("{:4} NETWORK ENGINEER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_network_eng, 
    np.array(salary_network_eng).mean(), np.array(salary_network_eng).min(), np.array(salary_network_eng).max()))
print("{:4} NETWORK / SECURITY SPEC.     | {:10.2f} | {:10.2f} | {:10.2f}".format(title_net_sec_specialist, 
    np.array(salary_net_sec_specialist).mean(), np.array(salary_net_sec_specialist).min(), np.array(salary_net_sec_specialist).max()))
print("{:4} OPERATIONS ANALYST           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_operations_analyst, 
    np.array(salary_operations_analyst).mean(), np.array(salary_operations_analyst).min(), np.array(salary_operations_analyst).max()))
print("{:4} PARTNER SPECIALIST           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_partner_spec, 
    np.array(salary_partner_spec).mean(), np.array(salary_partner_spec).min(), np.array(salary_partner_spec).max()))
print("{:4} PETROELUM ENGINEER           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_petroleum_eng, 
    np.array(salary_petroleum_eng).mean(), np.array(salary_petroleum_eng).min(), np.array(salary_petroleum_eng).max()))
print("{:4} PLATFORM ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_platform_eng, 
    np.array(salary_platform_eng).mean(), np.array(salary_platform_eng).min(), np.array(salary_platform_eng).max()))
print("{:4} PROCESS ENGINEER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_process_eng, 
    np.array(salary_process_eng).mean(), np.array(salary_process_eng).min(), np.array(salary_process_eng).max()))
print("{:4} PRODUCT MANAGER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_product_manager, 
    np.array(salary_product_manager).mean(), np.array(salary_product_manager).min(), np.array(salary_product_manager).max()))
print("{:4} PRODUCT / DIGITAL MARKETER   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_product_digital_mrkt, 
    np.array(salary_product_digital_mrkt).mean(), np.array(salary_product_digital_mrkt).min(), np.array(salary_product_digital_mrkt).max()))
print("{:4} PROGRAM MANAGER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_program_manager, 
    np.array(salary_program_manager).mean(), np.array(salary_program_manager).min(), np.array(salary_program_manager).max()))
print("{:4} PROJECT MANAGER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_project_manager, 
    np.array(salary_project_manager).mean(), np.array(salary_project_manager).min(), np.array(salary_project_manager).max()))
print("{:4} PYTHON DEVELOPER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_python_dev, 
    np.array(salary_python_dev).mean(), np.array(salary_python_dev).min(), np.array(salary_python_dev).max()))
print("{:4} QUALITY ASSURANCE ANALYST    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_qa_analyst, 
    np.array(salary_qa_analyst).mean(), np.array(salary_qa_analyst).min(), np.array(salary_qa_analyst).max()))
print("{:4} QUALITY ASSURANCE ENGINEER   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_qa_engineer, 
    np.array(salary_qa_engineer).mean(), np.array(salary_qa_engineer).min(), np.array(salary_qa_engineer).max()))
print("{:4} RESEARCH ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_research_eng, 
    np.array(salary_research_eng).mean(), np.array(salary_research_eng).min(), np.array(salary_research_eng).max()))
print("{:4} SALES DEVELOPMENT REPR.      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_sales_dev_repr, 
    np.array(salary_sales_dev_repr).mean(), np.array(salary_sales_dev_repr).min(), np.array(salary_sales_dev_repr).max()))
print("{:4} SALES ENGINEER               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_sales_eng, 
    np.array(salary_sales_eng).mean(), np.array(salary_sales_eng).min(), np.array(salary_sales_eng).max()))
print("{:4} SALESFORCE DEVELOPER         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_salesforce_dev, 
    np.array(salary_salesforce_dev).mean(), np.array(salary_salesforce_dev).min(), np.array(salary_salesforce_dev).max()))
print("{:4} SCRUM MASTER                 | {:10.2f} | {:10.2f} | {:10.2f}".format(title_scrum_master, 
    np.array(salary_scrum_master).mean(), np.array(salary_scrum_master).min(), np.array(salary_scrum_master).max()))
print("{:4} SECURITY ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_security_eng, 
    np.array(salary_security_eng).mean(), np.array(salary_security_eng).min(), np.array(salary_security_eng).max()))
print("{:4} SEO SPECIALIST               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_seo_specialist, 
    np.array(salary_seo_specialist).mean(), np.array(salary_seo_specialist).min(), np.array(salary_seo_specialist).max()))
print("{:4} SERVICENOW DEVELOPER         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_servicenow_dev, 
    np.array(salary_servicenow_dev).mean(), np.array(salary_servicenow_dev).min(), np.array(salary_servicenow_dev).max()))
print("{:4} SHAREPOINT DEVELOPER         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_sharepoint_dev, 
    np.array(salary_sharepoint_dev).mean(), np.array(salary_sharepoint_dev).min(), np.array(salary_sharepoint_dev).max()))
print("{:4} SOFTWARE ENGINEER            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_software_eng, 
    np.array(salary_software_eng).mean(), np.array(salary_software_eng).min(), np.array(salary_software_eng).max()))
print("{:4} SOLUTION ARCHITECT           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_solutions_architect, 
    np.array(salary_solutions_architect).mean(), np.array(salary_solutions_architect).min(), np.array(salary_solutions_architect).max()))
print("{:4} STAFF ENGINEER               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_staff_eng, 
    np.array(salary_staff_eng).mean(), np.array(salary_staff_eng).min(), np.array(salary_staff_eng).max()))
print("{:4} SUPPLY CHAIN SOLUTIONS ANLY. | {:10.2f} | {:10.2f} | {:10.2f}".format(title_supply_chain_sol, 
    np.array(salary_supply_chain_sol).mean(), np.array(salary_supply_chain_sol).min(), np.array(salary_supply_chain_sol).max()))
print("{:4} SUPPORT ENGINEER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_support_eng, 
    np.array(salary_support_eng).mean(), np.array(salary_support_eng).min(), np.array(salary_support_eng).max()))
print("{:4} SYSTEM ADMINISTRATOR         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_system_admin, 
    np.array(salary_system_admin).mean(), np.array(salary_system_admin).min(), np.array(salary_system_admin).max()))
print("{:4} SYSTEM ENGINEER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_system_eng, 
    np.array(salary_system_eng).mean(), np.array(salary_system_eng).min(), np.array(salary_system_eng).max()))
print("{:4} TECHNICAL ANALYST            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_technical_analyst, 
    np.array(salary_technical_analyst).mean(), np.array(salary_technical_analyst).min(), np.array(salary_technical_analyst).max()))
print("{:4} TEST ENGINEER                | {:10.2f} | {:10.2f} | {:10.2f}".format(title_test_eng, 
    np.array(salary_test_eng).mean(), np.array(salary_test_eng).min(), np.array(salary_test_eng).max()))
print("{:4} USER EXPERIENCE DESIGNER     | {:10.2f} | {:10.2f} | {:10.2f}".format(title_ux_ui_designer, 
    np.array(salary_ux_ui_designer).mean(), np.array(salary_ux_ui_designer).min(), np.array(salary_ux_ui_designer).max()))
print("{:4} WEB DEVELOPER                | {:10.2f} | {:10.2f} | {:10.2f}".format(title_web_developer, 
    np.array(salary_web_developer).mean(), np.array(salary_web_developer).min(), np.array(salary_web_developer).max()))
print("###### BLUE COLLAR TITLES ############# AVG ######## MIN ######## MAX ######")
print("{:4} ACADEMIC ADVISOR             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_academic_advisor, 
    np.array(salary_academic_advisor).mean(), np.array(salary_academic_advisor).min(), np.array(salary_academic_advisor).max()))
print("{:4} ACCOUNT EXECUTIVE            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_account_exec, 
    np.array(salary_account_exec).mean(), np.array(salary_account_exec).min(), np.array(salary_account_exec).max()))
print("{:4} ACCOUNT STRATEGIST           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_account_strategist, 
    np.array(salary_account_strategist).mean(), np.array(salary_account_strategist).min(), np.array(salary_account_strategist).max()))
print("{:4} ACCOUNTANT                   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_accountant, 
    np.array(salary_accountant).mean(), np.array(salary_accountant).min(), np.array(salary_accountant).max()))
print("{:4} ACTUARY                      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_actuary, 
    np.array(salary_actuary).mean(), np.array(salary_actuary).min(), np.array(salary_actuary).max()))
print("{:4} ADMISSION COUNSELOR          | {:10.2f} | {:10.2f} | {:10.2f}".format(title_admission_counselor, 
    np.array(salary_admission_counselor).mean(), np.array(salary_admission_counselor).min(), np.array(salary_admission_counselor).max()))
print("{:4} AUDITOR                      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_auditor, 
    np.array(salary_auditor).mean(), np.array(salary_auditor).min(), np.array(salary_auditor).max()))
print("{:4} BAKER                        | {:10.2f} | {:10.2f} | {:10.2f}".format(title_baker, 
    np.array(salary_baker).mean(), np.array(salary_baker).min(), np.array(salary_baker).max()))
print("{:4} BARISTA                      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_barista, 
    np.array(salary_barista).mean(), np.array(salary_barista).min(), np.array(salary_barista).max()))
print("{:4} BARTENDER                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_bartender, 
    np.array(salary_bartender).mean(), np.array(salary_bartender).min(), np.array(salary_bartender).max()))
print("{:4} BEAUTY ADVISOR               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_beauty_advisor, 
    np.array(salary_beauty_advisor).mean(), np.array(salary_beauty_advisor).min(), np.array(salary_beauty_advisor).max()))
print("{:4} BOOKKEEPER                   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_bookkeeper, 
    np.array(salary_bookkeeper).mean(), np.array(salary_bookkeeper).min(), np.array(salary_bookkeeper).max()))
print("{:4} CLAIMS ADJUSTER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_claims_adjuster, 
    np.array(salary_claims_adjuster).mean(), np.array(salary_claims_adjuster).min(), np.array(salary_claims_adjuster).max()))
print("{:4} COMPLIANCE OFFICER           | {:10.2f} | {:10.2f} | {:10.2f}".format(title_compliance_officer, 
    np.array(salary_compliance_officer).mean(), np.array(salary_compliance_officer).min(), np.array(salary_compliance_officer).max()))
print("{:4} CORPORATE COUNSEL            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_corporate_counsel, 
    np.array(salary_corporate_counsel).mean(), np.array(salary_corporate_counsel).min(), np.array(salary_corporate_counsel).max()))
print("{:4} CURRICULUM DEVELOPER         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_curriculum_dev, 
    np.array(salary_curriculum_dev).mean(), np.array(salary_curriculum_dev).min(), np.array(salary_curriculum_dev).max()))
print("{:4} CUSTOMER SERVICE MANAGER     | {:10.2f} | {:10.2f} | {:10.2f}".format(title_customer_serv_man, 
    np.array(salary_customer_serv_man).mean(), np.array(salary_customer_serv_man).min(), np.array(salary_customer_serv_man).max()))
print("{:4} DELIVERY DRIVER              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_delivery_driver, 
    np.array(salary_delivery_driver).mean(), np.array(salary_delivery_driver).min(), np.array(salary_delivery_driver).max()))
print("{:4} DENTAL ASSISTANT             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_dental_asst, 
    np.array(salary_dental_asst).mean(), np.array(salary_dental_asst).min(), np.array(salary_dental_asst).max()))
print("{:4} ELECTRICIAN                  | {:10.2f} | {:10.2f} | {:10.2f}".format(title_electrician, 
    np.array(salary_electrician).mean(), np.array(salary_electrician).min(), np.array(salary_electrician).max()))
print("{:4} EXECUTIVE ASSISTANT          | {:10.2f} | {:10.2f} | {:10.2f}".format(title_executive_asst, 
    np.array(salary_executive_asst).mean(), np.array(salary_executive_asst).min(), np.array(salary_executive_asst).max()))
print("{:4} FOOD SCIENTIST               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_food_scientist, 
    np.array(salary_food_scientist).mean(), np.array(salary_food_scientist).min(), np.array(salary_food_scientist).max()))
print("{:4} HELP DESK SUPPORT / ANALYST  | {:10.2f} | {:10.2f} | {:10.2f}".format(title_help_desk_support_anly, 
    np.array(salary_help_desk_support_anly).mean(), np.array(salary_help_desk_support_anly).min(), np.array(salary_help_desk_support_anly).max()))
print("{:4} INSIDE SALES                 | {:10.2f} | {:10.2f} | {:10.2f}".format(title_inside_sales, 
    np.array(salary_inside_sales).mean(), np.array(salary_inside_sales).min(), np.array(salary_inside_sales).max()))
print("{:4} INSURANCE AGENT              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_insurance_agent, 
    np.array(salary_insurance_agent).mean(), np.array(salary_insurance_agent).min(), np.array(salary_insurance_agent).max()))
print("{:4} INTERACTION DESIGNER         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_interaction_design, 
    np.array(salary_interaction_design).mean(), np.array(salary_interaction_design).min(), np.array(salary_interaction_design).max()))
print("{:4} LIBRARIAN                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_librarian, 
    np.array(salary_librarian).mean(), np.array(salary_librarian).min(), np.array(salary_librarian).max()))
print("{:4} MEDICAL ASSISTANT            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_medical_assistant, 
    np.array(salary_medical_assistant).mean(), np.array(salary_medical_assistant).min(), np.array(salary_medical_assistant).max()))
print("{:4} MERCHANDISER                 | {:10.2f} | {:10.2f} | {:10.2f}".format(title_merchandiser, 
    np.array(salary_merchandiser).mean(), np.array(salary_merchandiser).min(), np.array(salary_merchandiser).max()))
print("{:4} NURSE                        | {:10.2f} | {:10.2f} | {:10.2f}".format(title_nurse, 
    np.array(salary_nurse).mean(), np.array(salary_nurse).min(), np.array(salary_nurse).max()))
print("{:4} OFFICE ADMINISTRATOR         | {:10.2f} | {:10.2f} | {:10.2f}".format(title_office_admin, 
    np.array(salary_office_admin).mean(), np.array(salary_office_admin).min(), np.array(salary_office_admin).max()))
print("{:4} OUTSIDE SALES                | {:10.2f} | {:10.2f} | {:10.2f}".format(title_outside_sales, 
    np.array(salary_outside_sales).mean(), np.array(salary_outside_sales).min(), np.array(salary_outside_sales).max()))
print("{:4} PARALEGAL                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_paralegal, 
    np.array(salary_paralegal).mean(), np.array(salary_paralegal).min(), np.array(salary_paralegal).max()))
print("{:4} PARAMEDIC                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_paramedic, 
    np.array(salary_paramedic).mean(), np.array(salary_paramedic).min(), np.array(salary_paramedic).max()))
print("{:4} POLICE OFFICER               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_police_officer, 
    np.array(salary_police_officer).mean(), np.array(salary_police_officer).min(), np.array(salary_police_officer).max()))
print("{:4} POLICY ANALYST               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_policy_analyst, 
    np.array(salary_policy_analyst).mean(), np.array(salary_policy_analyst).min(), np.array(salary_policy_analyst).max()))
print("{:4} PRINCIPAL                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_principal, 
    np.array(salary_principal).mean(), np.array(salary_principal).min(), np.array(salary_principal).max()))
print("{:4} PROGRAM COORDINATOR          | {:10.2f} | {:10.2f} | {:10.2f}".format(title_program_coord, 
    np.array(salary_program_coord).mean(), np.array(salary_program_coord).min(), np.array(salary_program_coord).max()))
print("{:4} PSYCHOLOGIST                 | {:10.2f} | {:10.2f} | {:10.2f}".format(title_psychologist, 
    np.array(salary_psychologist).mean(), np.array(salary_psychologist).min(), np.array(salary_psychologist).max()))
print("{:4} REAL ESTATE AGENT            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_real_estate_agent, 
    np.array(salary_real_estate_agent).mean(), np.array(salary_real_estate_agent).min(), np.array(salary_real_estate_agent).max()))
print("{:4} RECRUIT                      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_recruit, 
    np.array(salary_recruit).mean(), np.array(salary_recruit).min(), np.array(salary_recruit).max()))
print("{:4} SALES ASSOCIATE              | {:10.2f} | {:10.2f} | {:10.2f}".format(title_sales_assoc, 
    np.array(salary_sales_assoc).mean(), np.array(salary_sales_assoc).min(), np.array(salary_sales_assoc).max()))
print("{:4} SCHOOL COUNSELOR             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_school_counselor, 
    np.array(salary_school_counselor).mean(), np.array(salary_school_counselor).min(), np.array(salary_school_counselor).max()))
print("{:4} SECRETARY                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_secretary, 
    np.array(salary_secretary).mean(), np.array(salary_secretary).min(), np.array(salary_secretary).max()))
print("{:4} SECURITY OFFICER             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_security_officer, 
    np.array(salary_security_officer).mean(), np.array(salary_security_officer).min(), np.array(salary_security_officer).max()))
print("{:4} SOCIAL WORKER                | {:10.2f} | {:10.2f} | {:10.2f}".format(title_social_worker, 
    np.array(salary_social_worker).mean(), np.array(salary_social_worker).min(), np.array(salary_social_worker).max()))
print("{:4} SUPERVISOR                   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_supervisor, 
    np.array(salary_supervisor).mean(), np.array(salary_supervisor).min(), np.array(salary_supervisor).max()))
print("{:4} TALENT ASSISTANT             | {:10.2f} | {:10.2f} | {:10.2f}".format(title_talent_asst, 
    np.array(salary_talent_asst).mean(), np.array(salary_talent_asst).min(), np.array(salary_talent_asst).max()))
print("{:4} TEACHER                      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_teacher, 
    np.array(salary_teacher).mean(), np.array(salary_teacher).min(), np.array(salary_teacher).max()))
print("{:4} TECHNICIAN                   | {:10.2f} | {:10.2f} | {:10.2f}".format(title_technician, 
    np.array(salary_technician).mean(), np.array(salary_technician).min(), np.array(salary_technician).max()))
print("{:4} THERAPIST                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_therapist, 
    np.array(salary_therapist).mean(), np.array(salary_therapist).min(), np.array(salary_therapist).max()))
print("{:4} TOURING ASSISTANT            | {:10.2f} | {:10.2f} | {:10.2f}".format(title_touring_asst, 
    np.array(salary_touring_asst).mean(), np.array(salary_touring_asst).min(), np.array(salary_touring_asst).max()))
print("{:4} TRANSPORTATION PLANNER       | {:10.2f} | {:10.2f} | {:10.2f}".format(title_transport_planner, 
    np.array(salary_transport_planner).mean(), np.array(salary_transport_planner).min(), np.array(salary_transport_planner).max()))
print("{:4} UNDERWRITER                  | {:10.2f} | {:10.2f} | {:10.2f}".format(title_underwriter, 
    np.array(salary_underwriter).mean(), np.array(salary_underwriter).min(), np.array(salary_underwriter).max()))
print("{:4} URBAN PLANNER                | {:10.2f} | {:10.2f} | {:10.2f}".format(title_urban_planner, 
    np.array(salary_urban_planner).mean(), np.array(salary_urban_planner).min(), np.array(salary_urban_planner).max()))
print("{:4} VICE PRESIDENT               | {:10.2f} | {:10.2f} | {:10.2f}".format(title_vice_president, 
    np.array(salary_vice_president).mean(), np.array(salary_vice_president).min(), np.array(salary_vice_president).max()))
print("{:4} WELDER                       | {:10.2f} | {:10.2f} | {:10.2f}".format(title_welder, 
    np.array(salary_welder).mean(), np.array(salary_welder).min(), np.array(salary_welder).max()))
print("###### OTHER NON-FITTING TITLES ####### AVG ######## MIN ######## MAX ######")
print("{:4} STUDENT                      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_student, 
    np.array(salary_student).mean(), np.array(salary_student).min(), np.array(salary_student).max()))
print("{:4} DEVELOPER                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_developer, 
    np.array(salary_developer).mean(), np.array(salary_developer).min(), np.array(salary_developer).max()))
print("{:4} ENGINEER                     | {:10.2f} | {:10.2f} | {:10.2f}".format(title_engineer, 
    np.array(salary_engineer).mean(), np.array(salary_engineer).min(), np.array(salary_engineer).max()))
print("{:4} SERVER                       | {:10.2f} | {:10.2f} | {:10.2f}".format(title_server, 
    np.array(salary_server).mean(), np.array(salary_server).min(), np.array(salary_server).max()))
print("{:4} ASSISTANT                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_assistant, 
    np.array(salary_assistant).mean(), np.array(salary_assistant).min(), np.array(salary_assistant).max()))
print("{:4} MARKET                       | {:10.2f} | {:10.2f} | {:10.2f}".format(title_market, 
    np.array(salary_market).mean(), np.array(salary_market).min(), np.array(salary_market).max()))
print("{:4} ANALYTICS                    | {:10.2f} | {:10.2f} | {:10.2f}".format(title_analytics, 
    np.array(salary_analytics).mean(), np.array(salary_analytics).min(), np.array(salary_analytics).max()))
print("{:4} ANALYST                      | {:10.2f} | {:10.2f} | {:10.2f}".format(title_analyst, 
    np.array(salary_analyst).mean(), np.array(salary_analyst).min(), np.array(salary_analyst).max()))
print("{:4} DESIGNER                     | {:10.2f} | {:10.2f} | {:10.2f}".format(title_designer, 
    np.array(salary_designer).mean(), np.array(salary_designer).min(), np.array(salary_designer).max()))
print("###### {:4} UNCATEGORIZED TITLES ######".format(title_uncategorized))

######   12 NULL TITLES ######
######   40 TITLES OPEN TO WORK / OPPORTUNITIES ######
###### WHITE COLLAR TITLES ############ AVG ######## MIN ######## MAX ######
  15 ACCOUNT MANAGER              |  219000.00 |   87000.00 |  310000.00
  22 AI ENGINEER                  |  172212.09 |   75000.00 |  256000.00
 477 APPLICATION DEV. / ENGINEER  |  174621.73 |   43517.00 |  386000.00
 187 APPLIED SCIENTIST            |  251460.57 |   74000.00 |  413000.00
  24 ASIC DESIGN ENGINEER         |  287563.21 |  200517.00 |  380000.00
  26 BACKEND DEVELOPER            |  173653.85 |  105000.00 |  260000.00
 549 BUSINESS ANALYST             |  126472.79 |   45390.00 |  265000.00
 172 BUSINESS INTELLIGENCE ENG.   |  181554.76 |   70706.00 |  235000.00
  48 CHEMICAL ENGINEER            |  115500.00 |   77000.00 |  193000.00
 218 CIVIL ENGINEER               |  114601.15 |   44051.00 |  178000.00
  33 CLOUD ENGINEER               |  132536.61 |   42845.00 |  420000.00
  23 COMPUTER ENGINEER            

## SALARY INFO BY TITLES (LIST INDEXES)

In [13]:
### Count each identified title of all rows, and collect their corresponding salaries
### Null titles and those indicating openness to work and opportunities
title_null = 0;           salary_null = [];
title_open_to_work = 0;   salary_open_to_work = [];  
### White collar titles in the list
white_collar_titles = ['Account Manager','AI Engineer','Application Developer / Engineer','Applied Scientist','ASIC Design Engineer',
    'Backend Developer','Business Analyst','Business Intelligence Engineer','Chemical Engineer','Civil Engineer','Cloud Engineer',
    'Computer Engineer','Consultant','Content Designer','Content Marketer','Content Strategist','Customer Engineer','Data Analyst',
    'Data Architect','Data Engineer','Data Scientist','Database Administrator','Database Engineer','Design Engineer','DevOps Engineer',
    'Distinguished Engineer','.NET Developer','E-Commerce','Electrical / Electronic Engineer','Energy Analyst','Environmental Engineer',
    'Financial Analyst','Firmware Developer','Frontend Developer','Fullstack Developer','Graphic Designer','Hardware Engineer',
    'Investment Banking Analyst','IT Architect / Engineer','Java Developer','Machine Learning Engineer','Marketing Analyst',
    'Mechanical Engineer','Mobile Developer','Network Administrator','Network Engineer','Network / Security Specialist',
    'Operations Analyst','Partner Specialist','Petroleum Engineer','Platform Engineer','Process Engineer','Product Manager',
    'Product / Digital Marketer','Program Manager','Project Manager','Python Developer','Quality Assurance Analyst',
    'Quality Assurance Engineer','Research Engineer','Sales Development Representative','Sales Engineer','Salesforce Developer',
    'Scrum Master','Security Engineer','SEO Specialist','ServiceNow Developer','Sharepoint Developer','Software Engineer',
    'Solution Architect','Staff Engineer','Supply Chain Solutions Analyst','Support Engineer','System Administrator','System Engineer',
    'Technical Analyst','Test Engineer','User Experience Designer','Web Developer']
print(len(white_collar_titles), "WHITE COLLAR TITLES")
white_collar_titles_cnt = [0 for _ in range(len(white_collar_titles))]
white_collar_titles_slr = [[] for _ in range(len(white_collar_titles))]
# print(white_collar_titles_cnt)
# print(white_collar_titles_slr)
### Blue collar titles in the list
blue_collar_titles = ['Academic Advisor','Account Executive','Account Strategist','Accountant','Actuary','Admission Counselor',
    'Auditor','Baker','Barista','Bartender','Beauty Advisor','Bookkeeper','Claims Adjuster','Compliance Officer','Corporate Counsel',
    'Curriculum Developer','Customer Service Manager','Delivery Driver','Dental Assistant','Electrician','Executive Assistant',
    'Food Scientist','Help Desk Support / Analyst','Inside Sales','Insurance Agent','Interaction Designer','Librarian',
    'Medical Assistant','Merchandiser','Nurse','Office Administrator','Outside Sales','Paralegal','Paramedic','Police Officer',
    'Policy Analyst','Principal','Program Coordinator','Psychologist','Real Estate Agent','Recruit','Sales Associate','School Counselor',
    'Secretary','Security Officer','Social Worker','Supervisor','Talent Assistant','Teacher','Technician','Therapist',
    'Touring Assistant','Transportation Planner','Underwriter','Urban Planner','Vice President','Welder']
print(len(blue_collar_titles), "BLUE COLLAR TITLES")
blue_collar_titles_cnt = [0 for _ in range(len(blue_collar_titles))]
blue_collar_titles_slr = [[] for _ in range(len(blue_collar_titles))]
# print(blue_collar_titles_cnt)
# print(blue_collar_titles_slr)
### Other titles in the list, that do not fit well in appropriate titles
other_titles = ['Student (Other)','Developer (Other)','Engineer (Other)','Server (Other)','Assistant (Other)','Marketer (Other)',
                'Analytics (Other)','Analyst (Other)','Designer (Other)']
print(len(other_titles), "OTHER TITLES")
other_titles_cnt = [0 for _ in range(len(other_titles))]
other_titles_slr = [[] for _ in range(len(other_titles))]
# print(other_titles_cnt)
# print(other_titles_slr)
# Uncategorized number of titles
title_uncategorized = 0;   salary_uncategorized = [];   title_uncategorized_examples = [];

79 WHITE COLLAR TITLES
57 BLUE COLLAR TITLES
9 OTHER TITLES


In [14]:
all_titles = salaries['title'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_titles)):
    title = all_titles[i];   salary = all_salaries[i]
    ### NULL TITLES ###
    if type(title) == float:   
        title_null += 1;   salary_null.append(salary);   continue   
    title = title.lower()
    ### OPEN TO WORK ###
    if 'open to work' in title or 'open to opportunities' in title or 'open to new opportunities' in title \
      or 'open to entry level' in title or 'looking for a new opportunity' in title or 'employment opportunity' in title \
      or 'experience with every given opportunity' in title or 'looking for new opportunities' in title \
      or 'looking for new carrier' in title or 'looking for opportunities' in title or 'looking for a new path' in title \
      or 'looking for a long term career' in title or 'looking for new recruiting opportunities' in title \
      or 'looking for professional opportunities' in title or 'looking for a full time position' in title \
      or 'looking for writing opportunities' in title or 'job searching' in title or 'seeking new opportunities' in title \
      or 'seeking opportunities' in title:
        title_open_to_work += 1;   salary_open_to_work.append(salary);
    ### WHITE COLLAR ###
    elif 'account manager' in title:   
        white_collar_titles_cnt[0] += 1;   white_collar_titles_slr[0].append(salary);
    elif 'ai engineering' in title or 'ai engineer' in title or 'ai frameworks engineer' in title \
      or 'ai research engineer' in title or 'ai model designer' in title or 'ai product designer' in title or 'ai designer' in title:
        white_collar_titles_cnt[1] += 1;   white_collar_titles_slr[1].append(salary);
    elif 'application developer' in title or 'app developer' in title or 'development engineer' in title \
      or 'application engineer' in title or 'applications engineer' in title:
        white_collar_titles_cnt[2] += 1;   white_collar_titles_slr[2].append(salary);
    elif 'applied scientist' in title:   
        white_collar_titles_cnt[3] += 1;   white_collar_titles_slr[3].append(salary);
    elif 'asic design engineer' in title:   
        white_collar_titles_cnt[4] += 1;   white_collar_titles_slr[4].append(salary);
    elif 'backend developer' in title or 'back end developer' in title or 'back-end developer' in title \
      or 'backend engineer' in title or 'back end engineer' in title or 'back-end engineer' in title:   
        white_collar_titles_cnt[5] += 1;   white_collar_titles_slr[5].append(salary);
    elif 'business analyst' in title or 'business intelligence analyst' in title or 'business technology analyst' in title \
      or 'business systems analyst' in title or 'business strategy analyst' in title or 'bi analyst' in title \
      or 'business analytics' in title or ('business' in title and 'analyst' in title):
        white_collar_titles_cnt[6] += 1;   white_collar_titles_slr[6].append(salary);
    elif 'business intelligence engineer' in title or 'bi engineer' in title or 'bi developer' in title \
      or 'business intelligence developer' in title or 'bi and analytics engineer' in title or 'business intelligence' in title:
        white_collar_titles_cnt[7] += 1;   white_collar_titles_slr[7].append(salary);
    elif 'chemical engineer' in title:   
        white_collar_titles_cnt[8] += 1;   white_collar_titles_slr[8].append(salary);
    elif 'civil engineer' in title or 'civil design engineer' in title or 'cilvil engineer' in title \
      or 'civil/structural/transportation engineer' in title:
        white_collar_titles_cnt[9] += 1;   white_collar_titles_slr[9].append(salary);
    elif 'cloud engineer' in title or 'cloud infrastructure engineer' in title or 'cloud and innovation engineer' in title \
      or 'cloud architect' in title:
        white_collar_titles_cnt[10] += 1;   white_collar_titles_slr[10].append(salary);
    elif 'computer engineer' in title:   
        white_collar_titles_cnt[11] += 1;   white_collar_titles_slr[11].append(salary);
    elif 'consultant' in title:   
        white_collar_titles_cnt[12] += 1;   white_collar_titles_slr[12].append(salary);
    elif 'content designer' in title or 'content developer' in title or 'content producer' in title:   
        white_collar_titles_cnt[13] += 1;   white_collar_titles_slr[13].append(salary);
    elif 'content marketing' in title or 'content marketer' in title:   
        white_collar_titles_cnt[14] += 1;   white_collar_titles_slr[14].append(salary);
    elif 'content strategist' in title:   
        white_collar_titles_cnt[15] += 1;   white_collar_titles_slr[15].append(salary);
    elif 'customer engineer' in title:   
        white_collar_titles_cnt[16] += 1;   white_collar_titles_slr[16].append(salary);
    elif 'data analyst' in title or 'data insights analyst' in title or 'data and systems analyst' in title \
      or 'data intelligence analyst' in title or 'data quality analyst' in title or 'data solutions analyst' in title \
      or 'data analytics' in title or ('data' in title and 'analyst' in title) or ('data' in title and 'analytics' in title):   
        white_collar_titles_cnt[17] += 1;   white_collar_titles_slr[17].append(salary);
    elif 'data architect' in title:   
        white_collar_titles_cnt[18] += 1;   white_collar_titles_slr[18].append(salary);
    elif 'data engineer' in title:   
        white_collar_titles_cnt[19] += 1;   white_collar_titles_slr[19].append(salary);
    elif 'data scientist' in title or 'data science' in title:   
        white_collar_titles_cnt[20] += 1;   white_collar_titles_slr[20].append(salary);
    elif 'database administrator' in title or 'database administration' in title or 'dba' in title:   
        white_collar_titles_cnt[21] += 1;   white_collar_titles_slr[21].append(salary);
    elif 'database engineer' in title:  
        white_collar_titles_cnt[22] += 1;   white_collar_titles_slr[22].append(salary);
    elif 'design engineer' in title or 'design verification engineer' in title or 'design/debug engineer' in title:   
        white_collar_titles_cnt[23] += 1;   white_collar_titles_slr[23].append(salary);
    elif 'devops engineer' in title or 'devsecops engineer' in title or 'devops' in title:   
        white_collar_titles_cnt[24] += 1;   white_collar_titles_slr[24].append(salary);
    elif 'distinguished engineer' in title:   
        white_collar_titles_cnt[25] += 1;   white_collar_titles_slr[25].append(salary);
    elif 'dotnet developer' in title or '.net developer' in title:   
        white_collar_titles_cnt[26] += 1;   white_collar_titles_slr[26].append(salary);
    elif 'ecommerce' in title or 'e-commerce' in title:   
        white_collar_titles_cnt[27] += 1;   white_collar_titles_slr[27].append(salary);
    elif 'electrical engineer' in title or 'electronic engineer' in title or 'electronics engineer' in title:   
        white_collar_titles_cnt[28] += 1;   white_collar_titles_slr[28].append(salary);
    elif 'energy analyst' in title or 'energy & chemicals' in title:   
        white_collar_titles_cnt[29] += 1;   white_collar_titles_slr[29].append(salary);
    elif 'environmental engineer' in title or 'environmental scientist' in title or 'environmental compliance engineer' in title:
        white_collar_titles_cnt[30] += 1;   white_collar_titles_slr[30].append(salary);
    elif 'financial analyst' in title or 'finance project analyst' in title or 'financial analytics' in title \
      or 'finance analyst' in title or 'finance professional' in title:
        white_collar_titles_cnt[31] += 1;   white_collar_titles_slr[31].append(salary);
    elif 'firmware developer' in title or 'firmware engineer' in title:   
        white_collar_titles_cnt[32] += 1;   white_collar_titles_slr[32].append(salary);
    elif 'frontend developer' in title or 'front-end developer' in title or 'front end developer' in title \
      or 'frontend engineer' in title or 'front-end engineer' in title or 'front end engineer' in title \
      or 'javascript developer' in title or 'react developer' in title:   
        white_collar_titles_cnt[33] += 1;   white_collar_titles_slr[33].append(salary);
    elif 'fullstack developer' in title or 'full stack developer' in title or 'full-stack developer' in title \
      or 'fullstack engineer' in title or 'full stack engineer' in title or 'full-stack engineer' in title:   
        white_collar_titles_cnt[34] += 1;   white_collar_titles_slr[34].append(salary);
    elif 'graphic designer' in title:   
        white_collar_titles_cnt[35] += 1;   white_collar_titles_slr[35].append(salary);
    elif 'hardware engineer' in title or 'hardware design engineer' in title:   
        white_collar_titles_cnt[36] += 1;   white_collar_titles_slr[36].append(salary);
    elif 'investment banking analyst' in title:   
        white_collar_titles_cnt[37] += 1;   white_collar_titles_slr[37].append(salary);
    elif 'it engineer' in title or 'it support engineer' in title or 'information technology engineer' in title \
      or 'it architect' in title:
        white_collar_titles_cnt[38] += 1;   white_collar_titles_slr[38].append(salary);
    elif 'java developer' in title or 'java/guidewire developer' in title \
      or 'java full stack aws developer' in title or 'java microservices developer' in title or 'java/j2ee developer' in title:   
        white_collar_titles_cnt[39] += 1;   white_collar_titles_slr[39].append(salary);
    elif 'machine learning engineer' in title or 'machine learning scientist' in title or 'ml engineer' in title \
      or 'machine learning' in title or 'deep learning engineer' in title or title.startswith('ml'):
        white_collar_titles_cnt[40] += 1;   white_collar_titles_slr[40].append(salary);
    elif 'marketing analyst' in title or 'market research analyst' in title or 'market analyst' in title \
      or 'marketing specialist' in title or 'marketing strategist' in title or 'marketing analytics' in title \
      or 'marketing execution analyst' in title or 'marketing executive' in title or 'market expert' in title \
      or 'marketing professional' in title or 'growth marketing' in title or 'brand marketing' in title \
      or 'global marketing' in title or 'salesforce marketing' in title or 'lifecycle marketing' in title:
        white_collar_titles_cnt[41] += 1;   white_collar_titles_slr[41].append(salary);
    elif 'mechanical engineer' in title or 'mechanical display engineer' in title or 'mechanical metrology engineer' in title:
        white_collar_titles_cnt[42] += 1;   white_collar_titles_slr[42].append(salary);
    elif 'mobile developer' in title or 'ios developer' in title or 'android developer' in title:   
        white_collar_titles_cnt[43] += 1;   white_collar_titles_slr[43].append(salary);
    elif 'network administrator' in title or 'network administration' in title:   
        white_collar_titles_cnt[44] += 1;   white_collar_titles_slr[44].append(salary);
    elif 'network engineer' in title or 'networks engineer' in title or 'network development engineer' in title \
      or 'network.engineer' in title or 'infrastructure engineer' in title or 'infrastructure services engineer' in title \
      or 'network consulting engineer' in title:
        white_collar_titles_cnt[45] += 1;   white_collar_titles_slr[45].append(salary);
    elif 'network specialist' in title or 'security specialist' in title:   
        white_collar_titles_cnt[46] += 1;   white_collar_titles_slr[46].append(salary);
    elif 'operations analyst' in title or 'operations engineer' in title:  
        white_collar_titles_cnt[47] += 1;   white_collar_titles_slr[47].append(salary);
    elif 'partner specialist' in title:   
        white_collar_titles_cnt[48] += 1;   white_collar_titles_slr[48].append(salary);
    elif 'petroleum engineer' in title:   
        white_collar_titles_cnt[49] += 1;   white_collar_titles_slr[49].append(salary);
    elif 'platform engineer' in title or 'platform developer' in title or 'plaform engineer' in title:   
        white_collar_titles_cnt[50] += 1;   white_collar_titles_slr[50].append(salary);
    elif 'process engineer' in title:   
        white_collar_titles_cnt[51] += 1;   white_collar_titles_slr[51].append(salary);
    elif 'product manager' in title or 'product development engineer' in title or 'product marketing manager' in title \
      or 'product analytics' in title:
        white_collar_titles_cnt[52] += 1;   white_collar_titles_slr[52].append(salary);
    elif 'product marketing' in title or 'product marketer' in title or 'digital marketing' in title \
      or 'digital marketer' in title or 'digital advertising' in title:   
        white_collar_titles_cnt[53] += 1;   white_collar_titles_slr[53].append(salary);
    elif 'program manager' in title:   
        white_collar_titles_cnt[54] += 1;   white_collar_titles_slr[54].append(salary);
    elif 'project manager' in title or 'project mananger' in title:   
        white_collar_titles_cnt[55] += 1;   white_collar_titles_slr[55].append(salary);
    elif 'python developer' in title:   
        white_collar_titles_cnt[56] += 1;   white_collar_titles_slr[56].append(salary);
    elif 'qa analyst' in title:   
        white_collar_titles_cnt[57] += 1;   white_collar_titles_slr[57].append(salary);
    elif 'qa engineer' in title or 'quality assurance' in title or 'quality engineer' in title \
      or 'qa automation engineer' in title or 'qa lead automation engineer' in title or 'qa mobile tester engineer' in title \
      or 'qa lead engineer' in title or 'qe engineering' in title or 'automation verification engineer' in title:
        white_collar_titles_cnt[58] += 1;   white_collar_titles_slr[58].append(salary);
    elif 'research engineer' in title:   
        white_collar_titles_cnt[59] += 1;   white_collar_titles_slr[59].append(salary);
    elif 'sales development representative' in title:   
        white_collar_titles_cnt[60] += 1;   white_collar_titles_slr[60].append(salary);
    elif 'sales engineer' in title:   
        white_collar_titles_cnt[61] += 1;   white_collar_titles_slr[61].append(salary);
    elif 'salesforce developer' in title or 'salesforce admin and developer' in title:   
        white_collar_titles_cnt[62] += 1;   white_collar_titles_slr[62].append(salary);
    elif 'scrum master' in title:   
        white_collar_titles_cnt[63] += 1;   white_collar_titles_slr[63].append(salary);
    elif 'security engineer' in title:   
        white_collar_titles_cnt[64] += 1;   white_collar_titles_slr[64].append(salary);
    elif 'seo specialist' in title or 'seo analyst' in title:   
        white_collar_titles_cnt[65] += 1;   white_collar_titles_slr[65].append(salary);
    elif 'servicenow developer' in title or 'servicenow grc/irm-secops architect' in title:   
        white_collar_titles_cnt[66] += 1;   white_collar_titles_slr[66].append(salary);
    elif 'sharepoint developer' in title:   
        white_collar_titles_cnt[67] += 1;   white_collar_titles_slr[67].append(salary);
    elif 'software engineer' in title or 'software development engineer' in title or 'software developer' in title \
      or 'software / firmware engineer' in title or 'softwear developer' in title or 'sofware developer' in title \
      or 'software dev engineer' in title or 'software devlopment engineer' in title \
      or 'software application development engineer' in title or 'software applications development engineer' in title \
      or 'software development emulator' in title or title.startswith('sde'):   
        white_collar_titles_cnt[68] += 1;   white_collar_titles_slr[68].append(salary);
    elif 'solution architect' in title or 'solutions architect' in title or 'solution developer' in title \
      or 'solution engineer' in title or 'solutions engineer' in title:
        white_collar_titles_cnt[69] += 1;   white_collar_titles_slr[69].append(salary);
    elif 'staff engineer' in title:   
        white_collar_titles_cnt[70] += 1;   white_collar_titles_slr[70].append(salary);
    elif 'supply chain solutions' in title or 'supply chain planning' in title or 'supply chain analyst' in title \
      or 'supply chain program analyst' in title:
        white_collar_titles_cnt[71] += 1;   white_collar_titles_slr[71].append(salary);
    elif 'support engineer' in title:   
        white_collar_titles_cnt[72] += 1;   white_collar_titles_slr[72].append(salary);
    elif 'system administrator' in title or 'systems administrator' in title:   
        white_collar_titles_cnt[73] += 1;   white_collar_titles_slr[73].append(salary);
    elif 'system engineer' in title or 'systems engineer' in title or 'system admin/engineer' in title \
      or 'system development engineer' in title or 'systems development engineer' in title:   
        white_collar_titles_cnt[74] += 1;   white_collar_titles_slr[74].append(salary);
    elif 'technical analyst' in title or 'technical solution analyst' in title or 'technical solutions analyst' in title:
        white_collar_titles_cnt[75] += 1;   white_collar_titles_slr[75].append(salary);
    elif 'test engineer' in title or 'test automation engineer' in title or 'test automation developer' in title \
      or 'automation engineer' in title or 'test development engineer' in title or 'test r&d engineer' in title \
      or 'test r & d engineer' in title or 'test product development engineer' in title or 'software test' in title:
        white_collar_titles_cnt[76] += 1;   white_collar_titles_slr[76].append(salary);
    elif 'user experience designer' in title or 'user experience, visual designer' in title \
      or 'user experience researcher & designer' in title or 'user experience engineer' in title \
      or 'user experience (ux) designer' in title or 'ux/ui designer' in title or 'ux ui designer' in title \
      or 'user experience researcher' in title or 'user experience & brand designer' in title \
      or 'ux visual & motion designer' in title or 'ui/ux visual designer' in title or 'ux/visual designer' in title \
      or 'ux designer' in title or 'ux engineer' in title or 'ux/ ui designer' in title or 'product designer' in title \
      or 'user interface developer' in title or 'ui developer' in title:   
        white_collar_titles_cnt[77] += 1;   white_collar_titles_slr[77].append(salary);
    elif 'web developer' in title or 'web production developer' in title:   
        white_collar_titles_cnt[78] += 1;   white_collar_titles_slr[78].append(salary);
    ### BLUE COLLAR ###
    elif 'academic advisor' in title or 'academic advising' in title:   
        blue_collar_titles_cnt[0] += 1;   blue_collar_titles_slr[0].append(salary);
    elif 'account executive' in title:   
        blue_collar_titles_cnt[1] += 1;   blue_collar_titles_slr[1].append(salary);
    elif 'account strategist' in title:   
        blue_collar_titles_cnt[2] += 1;   blue_collar_titles_slr[2].append(salary);
    elif 'accountant' in title:   
        blue_collar_titles_cnt[3] += 1;   blue_collar_titles_slr[3].append(salary);
    elif 'actuary' in title:   
        blue_collar_titles_cnt[4] += 1;   blue_collar_titles_slr[4].append(salary);
    elif 'admission counselor' in title or 'admissions counselor' in title:   
        blue_collar_titles_cnt[5] += 1;   blue_collar_titles_slr[5].append(salary);
    elif 'auditor' in title or 'audit associate' in title:   
        blue_collar_titles_cnt[6] += 1;   blue_collar_titles_slr[6].append(salary);
    elif 'baker' in title:   
        blue_collar_titles_cnt[7] += 1;   blue_collar_titles_slr[7].append(salary);
    elif 'barista' in title:   
        blue_collar_titles_cnt[8] += 1;   blue_collar_titles_slr[8].append(salary);
    elif 'bartender' in title:   
        blue_collar_titles_cnt[9] += 1;   blue_collar_titles_slr[9].append(salary);
    elif 'bookkeeper' in title:   
        blue_collar_titles_cnt[10] += 1;   blue_collar_titles_slr[10].append(salary);
    elif 'beauty advisor' in title:   
        blue_collar_titles_cnt[11] += 1;   blue_collar_titles_slr[11].append(salary);
    elif 'claims adjuster' in title:   
        blue_collar_titles_cnt[12] += 1;   blue_collar_titles_slr[12].append(salary);
    elif 'compliance officer' in title or 'compliance risk officer' in title or 'compliance/operations officer' in title \
      or 'compliance risk management officer' in title:
        blue_collar_titles_cnt[13] += 1;   blue_collar_titles_slr[13].append(salary);
    elif 'corporate counsel' in title:   
        blue_collar_titles_cnt[14] += 1;   blue_collar_titles_slr[14].append(salary);
    elif 'curriculum developer' in title:   
        blue_collar_titles_cnt[15] += 1;   blue_collar_titles_slr[15].append(salary);
    elif 'customer service manager' in title:   
        blue_collar_titles_cnt[16] += 1;   blue_collar_titles_slr[16].append(salary);
    elif 'delivery driver' in title:   
        blue_collar_titles_cnt[17] += 1;   blue_collar_titles_slr[17].append(salary);
    elif 'dental assistant' in title or 'dental assisting' in title:   
        blue_collar_titles_cnt[18] += 1;   blue_collar_titles_slr[18].append(salary);
    elif 'electrician' in title:   
        blue_collar_titles_cnt[19] += 1;   blue_collar_titles_slr[19].append(salary);
    elif 'executive assistant' in title or 'executive & administrative assistant' in title:   
        blue_collar_titles_cnt[20] += 1;   blue_collar_titles_slr[20].append(salary);
    elif 'food scientist' in title:   
        blue_collar_titles_cnt[21] += 1;   blue_collar_titles_slr[21].append(salary);
    elif 'help desk support' in title or 'help desk analyst' in title:   
        blue_collar_titles_cnt[22] += 1;   blue_collar_titles_slr[22].append(salary);
    elif 'inside sales' in title:   
        blue_collar_titles_cnt[23] += 1;   blue_collar_titles_slr[23].append(salary);
    elif 'insurance agent' in title:   
        blue_collar_titles_cnt[24] += 1;   blue_collar_titles_slr[24].append(salary);
    elif 'interaction designer' in title:   
        blue_collar_titles_cnt[25] += 1;   blue_collar_titles_slr[25].append(salary);
    elif 'librarian' in title:   
        blue_collar_titles_cnt[26] += 1;   blue_collar_titles_slr[26].append(salary);
    elif 'medical assistant' in title:   
        blue_collar_titles_cnt[27] += 1;   blue_collar_titles_slr[27].append(salary);
    elif 'merchandiser' in title:   
        blue_collar_titles_cnt[28] += 1;   blue_collar_titles_slr[28].append(salary);
    elif 'nurse' in title:   
        blue_collar_titles_cnt[29] += 1;   blue_collar_titles_slr[29].append(salary);
    elif 'office administrator' in title:   
        blue_collar_titles_cnt[30] += 1;   blue_collar_titles_slr[30].append(salary);
    elif 'outside sales' in title:   
        blue_collar_titles_cnt[31] += 1;   blue_collar_titles_slr[31].append(salary);
    elif 'paralegal' in title:   
        blue_collar_titles_cnt[32] += 1;   blue_collar_titles_slr[32].append(salary);
    elif 'paramedic' in title:   
        blue_collar_titles_cnt[33] += 1;   blue_collar_titles_slr[33].append(salary);
    elif 'police officer' in title or 'police official' in title:   
        blue_collar_titles_cnt[34] += 1;   blue_collar_titles_slr[34].append(salary);
    elif 'policy analyst' in title:   
        blue_collar_titles_cnt[35] += 1;   blue_collar_titles_slr[35].append(salary);
    elif 'principal' in title:   
        blue_collar_titles_cnt[36] += 1;   blue_collar_titles_slr[36].append(salary);
    elif 'program coordinator' in title:   
        blue_collar_titles_cnt[37] += 1;   blue_collar_titles_slr[37].append(salary);
    elif 'psychologist' in title:   
        blue_collar_titles_cnt[38] += 1;   blue_collar_titles_slr[38].append(salary);
    elif 'real estate agent' in title:   
        blue_collar_titles_cnt[39] += 1;   blue_collar_titles_slr[39].append(salary);
    elif 'recruit' in title:   
        blue_collar_titles_cnt[40] += 1;   blue_collar_titles_slr[40].append(salary);
    elif 'sales associate' in title or 'sales representative' in title:   
        blue_collar_titles_cnt[41] += 1;   blue_collar_titles_slr[41].append(salary);
    elif 'school counselor' in title:  
        blue_collar_titles_cnt[42] += 1;   blue_collar_titles_slr[42].append(salary);
    elif 'secretary' in title:   
        blue_collar_titles_cnt[43] += 1;   blue_collar_titles_slr[43].append(salary);
    elif 'security officer' in title or 'security guard' in title:   
        blue_collar_titles_cnt[44] += 1;   blue_collar_titles_slr[44].append(salary);
    elif 'social worker' in title or 'lmsw' in title:   
        blue_collar_titles_cnt[45] += 1;   blue_collar_titles_slr[45].append(salary);
    elif 'supervisor' in title:   
        blue_collar_titles_cnt[46] += 1;   blue_collar_titles_slr[46].append(salary);
    elif 'talent assistant' in title:   
        blue_collar_titles_cnt[47] += 1;   blue_collar_titles_slr[47].append(salary);
    elif 'teacher' in title:   
        blue_collar_titles_cnt[48] += 1;   blue_collar_titles_slr[48].append(salary);
    elif 'technician' in title:   
        blue_collar_titles_cnt[49] += 1;   blue_collar_titles_slr[49].append(salary);
    elif 'therapist' in title:   
        blue_collar_titles_cnt[50] += 1;   blue_collar_titles_slr[50].append(salary);
    elif 'touring assistant' in title:   
        blue_collar_titles_cnt[51] += 1;   blue_collar_titles_slr[51].append(salary);
    elif 'transportation planner' in title:   
        blue_collar_titles_cnt[52] += 1;   blue_collar_titles_slr[52].append(salary);
    elif 'underwriter' in title:   
        blue_collar_titles_cnt[53] += 1;   blue_collar_titles_slr[53].append(salary);
    elif 'urban planner' in title:   
        blue_collar_titles_cnt[54] += 1;   blue_collar_titles_slr[54].append(salary);
    elif 'vice president' in title:   
        blue_collar_titles_cnt[55] += 1;   blue_collar_titles_slr[55].append(salary);
    elif 'welder' in title:   
        blue_collar_titles_cnt[56] += 1;   blue_collar_titles_slr[56].append(salary);
    ### OTHER NON-FITTING TITLES ###
    elif 'student' in title:     other_titles_cnt[0] += 1;   other_titles_slr[0].append(salary);
    elif 'developer' in title:   other_titles_cnt[1] += 1;   other_titles_slr[1].append(salary);
    elif 'engineer' in title:    other_titles_cnt[2] += 1;   other_titles_slr[2].append(salary);
    elif 'server' in title:      other_titles_cnt[3] += 1;   other_titles_slr[3].append(salary);
    elif 'assistant' in title:   other_titles_cnt[4] += 1;   other_titles_slr[4].append(salary);
    elif 'market' in title:      other_titles_cnt[5] += 1;   other_titles_slr[5].append(salary);
    elif 'analytics' in title:   other_titles_cnt[6] += 1;   other_titles_slr[6].append(salary);
    elif 'analyst' in title:     other_titles_cnt[7] += 1;   other_titles_slr[7].append(salary);
    elif 'designer' in title:    other_titles_cnt[8] += 1;   other_titles_slr[8].append(salary);
    ### UNCATEGORIZED TITLES ###
    else:   title_uncategorized += 1;   salary_uncategorized.append(salary);   title_uncategorized_examples.append(title)

In [15]:
print("#### COUNTS OF ALL WHITE COLLAR TITLES ####")
print(white_collar_titles_cnt, '-->', sum(white_collar_titles_cnt))
print("#### COUNTS OF ALL BLUE COLLAR TITLES ####")
print(blue_collar_titles_cnt, '-->', sum(blue_collar_titles_cnt))
print("#### COUNTS OF ALL OTHER TITLES ####")
print(other_titles_cnt, '-->', sum(other_titles_cnt))
print("#### {} OPEN TO WORK TITLES ####".format(title_open_to_work))
print("#### {} UNCATEGORIZED TITLES ####".format(title_uncategorized))
print("#### {} NULL TITLES ####".format(title_null))

#### COUNTS OF ALL WHITE COLLAR TITLES ####
[15, 22, 477, 187, 24, 26, 549, 172, 48, 218, 33, 23, 169, 35, 10, 102, 25, 969, 20, 154, 1602, 85, 57, 88, 284, 18, 2, 20, 198, 19, 64, 167, 11, 150, 40, 76, 180, 16, 79, 19, 874, 48, 74, 12, 19, 379, 7, 28, 3, 29, 8, 19, 352, 29, 497, 296, 6, 6, 310, 5, 83, 138, 12, 5, 195, 21, 4, 7, 3787, 42, 5, 81, 130, 85, 120, 17, 273, 508, 32] --> 14999
#### COUNTS OF ALL BLUE COLLAR TITLES ####
[44, 184, 5, 60, 141, 56, 52, 8, 23, 17, 15, 31, 46, 48, 100, 26, 8, 19, 38, 83, 11, 17, 16, 92, 95, 9, 55, 101, 78, 185, 5, 106, 46, 19, 50, 32, 77, 78, 108, 115, 8, 99, 77, 44, 88, 95, 75, 4, 97, 207, 50, 4, 72, 103, 15, 41, 14] --> 3392
#### COUNTS OF ALL OTHER TITLES ####
[31, 18, 29, 6, 44, 21, 6, 27, 19] --> 201
#### 40 OPEN TO WORK TITLES ####
#### 909 UNCATEGORIZED TITLES ####
#### 12 NULL TITLES ####


In [16]:
print("############################ COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####")
print("NULL TITLES                 | {:4} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f}".format(
    title_null, np.array(salary_null).mean(), np.array(salary_null).std(), np.array(salary_null).min(), np.percentile(salary_null, 25),
    np.percentile(salary_null, 50), np.percentile(salary_null, 75), np.array(salary_null).max()))
print("TITLES OPEN TO WORK         | {:4} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f}".format(
    title_open_to_work, np.array(salary_open_to_work).mean(), np.array(salary_open_to_work).std(), np.array(salary_open_to_work).min(),
    np.percentile(salary_open_to_work, 25), np.percentile(salary_open_to_work, 50), np.percentile(salary_open_to_work, 75),
    np.array(salary_open_to_work).max()))
print("UNCATEGORIZED TITLES        | {:4} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f}".format(
    title_uncategorized, np.array(salary_uncategorized).mean(), np.array(salary_uncategorized).std(), np.array(salary_uncategorized).min(),
    np.percentile(salary_uncategorized, 25), np.percentile(salary_uncategorized, 50), np.percentile(salary_uncategorized, 75),
    np.array(salary_uncategorized).max()))

############################ COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####
NULL TITLES                 |   12 |  121268.08 |   56320.07 |   54000.00 |   78377.50 |   99500.00 |  171250.00 |  230000.00
TITLES OPEN TO WORK         |   40 |   87289.70 |   44570.38 |   27000.00 |   46567.50 |   86500.00 |  123250.00 |  218000.00
UNCATEGORIZED TITLES        |  909 |  117047.38 |   75505.44 |   21000.00 |   62000.00 |   97000.00 |  145000.00 |  540000.00


In [17]:
print("#### WHITE COLLAR TITLE ############ COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####")
for i in range(len(white_collar_titles)):
    print("{:35} | {:4} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f}".format(
        white_collar_titles[i].upper(), white_collar_titles_cnt[i], np.array(white_collar_titles_slr[i]).mean(),
        np.array(white_collar_titles_slr[i]).std(), np.array(white_collar_titles_slr[i]).min(), 
        np.percentile(white_collar_titles_slr[i], 25), np.percentile(white_collar_titles_slr[i], 50), 
        np.percentile(white_collar_titles_slr[i], 75), np.array(white_collar_titles_slr[i]).max()))

#### WHITE COLLAR TITLE ############ COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####
ACCOUNT MANAGER                     |   15 |  219000.00 |   67053.21 |   87000.00 |  168000.00 |  220000.00 |  280000.00 |  310000.00
AI ENGINEER                         |   22 |  172212.09 |   37315.66 |   75000.00 |  156250.00 |  166000.00 |  197499.50 |  256000.00
APPLICATION DEVELOPER / ENGINEER    |  477 |  174621.73 |   55557.60 |   43517.00 |  140000.00 |  180000.00 |  210000.00 |  386000.00
APPLIED SCIENTIST                   |  187 |  251460.57 |   65896.23 |   74000.00 |  200500.00 |  255000.00 |  294500.00 |  413000.00
ASIC DESIGN ENGINEER                |   24 |  287563.21 |   48555.68 |  200517.00 |  249250.00 |  283500.00 |  334000.00 |  380000.00
BACKEND DEVELOPER                   |   26 |  173653.85 |   25586.87 |  105000.00 |  160750.00 |  172000.00 |  184750.00 |  260000.00
BUSINESS ANALYST                    |  549 |  126472.79 |   

In [18]:
print("#### BLUE COLLAR TITLE ############# COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####")
for i in range(len(blue_collar_titles)):
    print("{:35} | {:4} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f}".format(
        blue_collar_titles[i].upper(), blue_collar_titles_cnt[i], np.array(blue_collar_titles_slr[i]).mean(),
        np.array(blue_collar_titles_slr[i]).std(), np.array(blue_collar_titles_slr[i]).min(),  
        np.percentile(blue_collar_titles_slr[i], 25), np.percentile(blue_collar_titles_slr[i], 50), 
        np.percentile(blue_collar_titles_slr[i], 75), np.array(blue_collar_titles_slr[i]).max()))

#### BLUE COLLAR TITLE ############# COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####
ACADEMIC ADVISOR                    |   44 |   76636.36 |   12310.55 |   49000.00 |   71500.00 |   79500.00 |   84000.00 |   93000.00
ACCOUNT EXECUTIVE                   |  184 |  142396.74 |   77218.07 |   50000.00 |   89500.00 |  102000.00 |  182750.00 |  345000.00
ACCOUNT STRATEGIST                  |    5 |  113600.00 |    6858.57 |  108000.00 |  108000.00 |  108000.00 |  122000.00 |  122000.00
ACCOUNTANT                          |   60 |  106400.00 |   21572.51 |   73000.00 |   90000.00 |  110000.00 |  110000.00 |  180000.00
ACTUARY                             |  141 |  217312.06 |   37494.61 |  112000.00 |  192000.00 |  220000.00 |  250000.00 |  275000.00
ADMISSION COUNSELOR                 |   56 |   76589.29 |   12545.31 |   47000.00 |   69250.00 |   78000.00 |   86000.00 |   95000.00
AUDITOR                             |   52 |   93442.31 |   

In [19]:
print("#### OTHER TITLE ################### COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####")
for i in range(len(other_titles)):
    print("{:35} | {:4} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f} | {:10.2f}".format(
        other_titles[i].upper(), other_titles_cnt[i], np.array(other_titles_slr[i]).mean(),
        np.array(other_titles_slr[i]).std(), np.array(other_titles_slr[i]).min(),  
        np.percentile(other_titles_slr[i], 25), np.percentile(other_titles_slr[i], 50), 
        np.percentile(other_titles_slr[i], 75), np.array(other_titles_slr[i]).max()))

#### OTHER TITLE ################### COUNT ##### AVG ######## STD ######## MIN ######## 25% ######## 50% ######## 75% ######## MAX ####
STUDENT (OTHER)                     |   31 |   62516.13 |   27535.16 |   34000.00 |   42500.00 |   48000.00 |   76000.00 |  125000.00
DEVELOPER (OTHER)                   |   18 |  152580.33 |   65003.54 |   41491.00 |  116500.00 |  142500.00 |  182750.00 |  278000.00
ENGINEER (OTHER)                    |   29 |  157409.28 |   58226.14 |   30140.00 |  134000.00 |  161000.00 |  189000.00 |  285000.00
SERVER (OTHER)                      |    6 |  106833.33 |   18068.54 |   86000.00 |   92000.00 |  108000.00 |  111250.00 |  140000.00
ASSISTANT (OTHER)                   |   44 |   54363.64 |   18824.66 |   32000.00 |   45750.00 |   49500.00 |   55000.00 |  142000.00
MARKETER (OTHER)                    |   21 |  118857.14 |   46859.76 |   57000.00 |   80000.00 |  110000.00 |  137000.00 |  242000.00
ANALYTICS (OTHER)                   |    6 |  186061.83 |   

In [20]:
salary_by_title = pd.DataFrame(columns=['Title','Count','Average','Minimum','25thPercent','50thPercent','75thPercent','Maximum'])
for i in range(len(white_collar_titles)):
    salary_row = pd.DataFrame({'Title': [white_collar_titles[i]], 
                  'Count': [white_collar_titles_cnt[i]], 
                  'Average': [np.array(white_collar_titles_slr[i]).mean()], 
                  'Minimum': [np.array(white_collar_titles_slr[i]).min()],
                  '25thPercent': [np.percentile(white_collar_titles_slr[i], 25)], 
                  '50thPercent': [np.percentile(white_collar_titles_slr[i], 50)],
                  '75thPercent': [np.percentile(white_collar_titles_slr[i], 75)],
                  'Maximum': [np.array(white_collar_titles_slr[i]).max()]})
    # salary_by_title = salary_by_title.append(salary_row, ignore_index=True)
    salary_by_title = pd.concat([salary_by_title, salary_row], ignore_index=True)
for i in range(len(blue_collar_titles)):
    salary_row = pd.DataFrame({'Title': [blue_collar_titles[i]], 
                  'Count': [blue_collar_titles_cnt[i]], 
                  'Average': [np.array(blue_collar_titles_slr[i]).mean()], 
                  'Minimum': [np.array(blue_collar_titles_slr[i]).min()],
                  '25thPercent': [np.percentile(blue_collar_titles_slr[i], 25)], 
                  '50thPercent': [np.percentile(blue_collar_titles_slr[i], 50)],
                  '75thPercent': [np.percentile(blue_collar_titles_slr[i], 75)],
                  'Maximum': [np.array(blue_collar_titles_slr[i]).max()]})
    salary_by_title = pd.concat([salary_by_title, salary_row], ignore_index=True)
for i in range(len(other_titles)):
    salary_row = pd.DataFrame({'Title': [other_titles[i]], 
                  'Count': [other_titles_cnt[i]], 
                  'Average': [np.array(other_titles_slr[i]).mean()], 
                  'Minimum': [np.array(other_titles_slr[i]).min()],
                  '25thPercent': [np.percentile(other_titles_slr[i], 25)], 
                  '50thPercent': [np.percentile(other_titles_slr[i], 50)],
                  '75thPercent': [np.percentile(other_titles_slr[i], 75)],
                  'Maximum': [np.array(other_titles_slr[i]).max()]})
    salary_by_title = pd.concat([salary_by_title, salary_row], ignore_index=True)

In [21]:
salary_by_title.head(10)

,Title,Count,Average,Minimum,25thPercent,50thPercent,75thPercent,Maximum
0,Account Manager,15,219000.000000,87000,168000.0,220000.0,280000.0,310000
1,AI Engineer,22,172212.090909,75000,156250.0,166000.0,197499.5,256000
2,Application Developer / Engineer,477,174621.729560,43517,140000.0,180000.0,210000.0,386000
3,Applied Scientist,187,251460.566845,74000,200500.0,255000.0,294500.0,413000
4,ASIC Design Engineer,24,287563.208333,200517,249250.0,283500.0,334000.0,380000
5,Backend Developer,26,173653.846154,105000,160750.0,172000.0,184750.0,260000
6,Business Analyst,549,126472.786885,45390,95000.0,120000.0,156000.0,265000
7,Business Intelligence Engineer,172,181554.761628,70706,170000.0,185000.0,205000.0,235000
8,Chemical Engineer,48,115500.000000,77000,89500.0,110000.0,137000.0,193000
9,Civil Engineer,218,114601.151376,44051,95000.0,110000.0,134000.0,178000


In [22]:
salary_by_title.tail(10)

,Title,Count,Average,Minimum,25thPercent,50thPercent,75thPercent,Maximum
135,Welder,14,65928.571429,50000,59000.0,67500.0,70250.0,79000
136,Student (Other),31,62516.129032,34000,42500.0,48000.0,76000.0,125000
137,Developer (Other),18,152580.333333,41491,116500.0,142500.0,182750.0,278000
138,Engineer (Other),29,157409.275862,30140,134000.0,161000.0,189000.0,285000
139,Server (Other),6,106833.333333,86000,92000.0,108000.0,111250.0,140000
140,Assistant (Other),44,54363.636364,32000,45750.0,49500.0,55000.0,142000
141,Marketer (Other),21,118857.142857,57000,80000.0,110000.0,137000.0,242000
142,Analytics (Other),6,186061.833333,108371,165000.0,178500.0,204000.0,278000
143,Analyst (Other),27,111085.962963,44000,83362.5,115000.0,133000.0,193000
144,Designer (Other),19,142105.263158,88000,116500.0,135000.0,170000.0,220000


In [23]:
salary_by_title.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        145 non-null    object 
 1   Count        145 non-null    object 
 2   Average      145 non-null    float64
 3   Minimum      145 non-null    object 
 4   25thPercent  145 non-null    float64
 5   50thPercent  145 non-null    float64
 6   75thPercent  145 non-null    float64
 7   Maximum      145 non-null    object 
dtypes: float64(4), object(4)
memory usage: 9.2+ KB


In [24]:
salary_by_title.to_csv('salaries_by_title.csv')

### GET SALARY INFO BY TITLE WITH COUNTRY COEF, ALSO FIND FOR SIMILAR ONES

In [25]:
titles_salaries = pd.read_csv('salaries_by_title.csv')
# countries_coef = pd.read_csv('countries_salary_coef.csv')
titles_salaries.drop('Unnamed: 0', axis=1, inplace=True)

In [26]:
titles_salaries

,Title,Count,Average,Minimum,25thPercent,50thPercent,75thPercent,Maximum
0,Account Manager,15,219000.000000,87000,168000.0,220000.0,280000.0,310000
1,AI Engineer,22,172212.090909,75000,156250.0,166000.0,197499.5,256000
2,Application Developer / Engineer,477,174621.729560,43517,140000.0,180000.0,210000.0,386000
3,Applied Scientist,187,251460.566845,74000,200500.0,255000.0,294500.0,413000
4,ASIC Design Engineer,24,287563.208333,200517,249250.0,283500.0,334000.0,380000
...,...,...,...,...,...,...,...,...
140,Assistant (Other),44,54363.636364,32000,45750.0,49500.0,55000.0,142000
141,Marketer (Other),21,118857.142857,57000,80000.0,110000.0,137000.0,242000
142,Analytics (Other),6,186061.833333,108371,165000.0,178500.0,204000.0,278000
143,Analyst (Other),27,111085.962963,44000,83362.5,115000.0,133000.0,193000


In [29]:
# TARGET TITLE EXAMPLES: "Account Manager", "Application Developer / Engineer", "Business Analyst", "Civil Engineer", "Data Analyst",
# "Data Scientist", "DevOps Engineer", "Financial Analyst", "Hardware Engineer", "IT Architect / Engineer", "Machine Learning Engineer", 
# "Network Engineer", "Quality Assurance Engineer", "SEO Specialist", "Software Engineer", "Test Engineer", "User Experience Designer",
# "Account Executive", "Actuary", "Electrician", "Librarian", "Outside Sales", "Secretary", "Underwriter"
target_title = "Data Analyst"
# TARGET COUNTRY EXAMPLES: "United States", "Canada", "Luxembourg", "Iceland", "Norway", "Germany", "Australia", "Portugal", "Spain"
# "Slovenia", "China", "France", "Denmark", "Finland", "Japan", "Sweden", "Russia", "Brazil", "Turkey", "India", "Peru", "Iraq",
# "Indonesia", "Egypt", "Libya", "Rwanda", "Mali", "Yemen"
target_country = "Luxembourg"

target_title_salary = titles_salaries[titles_salaries['Title'] == target_title]
target_country_coef = countries_coef[countries_coef['en_name'] == target_country]['salary_coefficient'].values
if not target_title_salary.any().any():
    print("NO SALARY INFO | TARGET TITLE \"{}\" WAS NOT FOUND".format(target_title))
elif not target_country_coef.any():
    print("NO SALARY INFO | TARGET COUNTRY \"{}\" WAS NOT FOUND".format(target_country))
elif np.isnan(target_country_coef)[0] == True:
    print("NO SALARY INFO | COEFFICIENT OF TARGET COUNTRY \"{}\" WAS NULL".format(target_country))
else:
    print("TARGET TITLE: {} | TARGET COUNTRY: {} ({:.3f})".format(target_title, target_country, target_country_coef[0]))
    print("TITLE COUNT >>       {:10}".format(target_title_salary['Count'].values[0]))
    print("AVERAGE SALARY >>    {:10.2f} ({:.2f})".format(
        target_title_salary['Average'].values[0], target_title_salary['Average'].values[0]*target_country_coef[0]))
    print("MINIMUM >>           {:10.2f} ({:.2f})".format(
        target_title_salary['Minimum'].values[0], target_title_salary['Minimum'].values[0]*target_country_coef[0]))
    print("25TH PERCENT >>      {:10.2f} ({:.2f})".format(
        target_title_salary['25thPercent'].values[0], target_title_salary['25thPercent'].values[0]*target_country_coef[0]))
    print("50TH PERCENT >>      {:10.2f} ({:.2f})".format(
        target_title_salary['50thPercent'].values[0], target_title_salary['50thPercent'].values[0]*target_country_coef[0]))
    print("75TH PERCENT >>      {:10.2f} ({:.2f})".format(
        target_title_salary['75thPercent'].values[0], target_title_salary['75thPercent'].values[0]*target_country_coef[0]))
    print("MAXIMUM >>           {:10.2f} ({:.2f})".format(
        target_title_salary['Maximum'].values[0], target_title_salary['Maximum'].values[0]*target_country_coef[0]))
    print("PROJECTED MINIMUM >> {:10.2f} ({:.2f})".format(
        target_title_salary['Average'].values[0]*0.8, target_title_salary['Average'].values[0]*target_country_coef[0]*0.8))
    print("PROJECTED MAXIMUM >> {:10.2f} ({:.2f})".format(
        target_title_salary['Average'].values[0]*1.2, target_title_salary['Average'].values[0]*target_country_coef[0]*1.2))

    similar_titles = []
    target_title_words = target_title.replace('/','').split()
    for tt in titles_salaries['Title']:
        if tt == target_title:
            continue
        for word in target_title_words:
            if word in tt:
                if tt not in similar_titles:
                    similar_titles.append(tt)
    print("\n{} SIMILAR TITLES AVAILABLE".format(len(similar_titles)))
    for st in similar_titles:
        similar_title_salary = titles_salaries[titles_salaries['Title'] == st]
        print("{:35}; {:.2f}, {:.2f}, {:.2f}, {:.2f}, {:.2f}, {:.2f}".format(
            similar_title_salary['Title'].values[0], 
            similar_title_salary['Average'].values[0]*target_country_coef[0], 
            similar_title_salary['Minimum'].values[0]*target_country_coef[0], 
            similar_title_salary['25thPercent'].values[0]*target_country_coef[0], 
            similar_title_salary['50thPercent'].values[0]*target_country_coef[0], 
            similar_title_salary['75thPercent'].values[0]*target_country_coef[0], 
            similar_title_salary['Maximum'].values[0]*target_country_coef[0]))

TARGET TITLE: Data Analyst | TARGET COUNTRY: Luxembourg (0.766)
TITLE COUNT >>              969
AVERAGE SALARY >>     129742.21 (99382.53)
MINIMUM >>             31818.00 (24372.59)
25TH PERCENT >>       101000.00 (77366.00)
50TH PERCENT >>       121000.00 (92686.00)
75TH PERCENT >>       148000.00 (113368.00)
MAXIMUM >>            414000.00 (317124.00)
PROJECTED MINIMUM >>  103793.77 (79506.02)
PROJECTED MAXIMUM >>  155690.65 (119259.04)

17 SIMILAR TITLES AVAILABLE
Business Analyst                   ; 96878.15, 34768.74, 72770.00, 91920.00, 119496.00, 202990.00
Data Architect                     ; 153505.75, 53880.44, 145348.50, 167754.00, 180010.00, 210650.00
Data Engineer                      ; 153417.59, 33151.71, 125815.50, 142859.00, 183074.00, 278824.00
Data Scientist                     ; 154137.67, 39832.00, 117964.00, 146306.00, 189202.00, 344700.00
Database Administrator             ; 117531.44, 49790.00, 99580.00, 114900.00, 130220.00, 206820.00
Database Engineer          